# VT Research Suite: Vanilla ResNet-18 Kaggle All-In-One

This notebook is a separate deterministic VT pipeline built around a vanilla ResNet-18-style SNN backbone.

Key choices:
- deterministic LIF neurons
- direct input encoding
- no stochastic noise
- no Parseval regularization
- non-EOT PGD-20 and standard AutoAttack
- full calibration and clean evaluation, with a class-balanced 2000-sample attack subset

In [ ]:
RUN_OPTIONAL_INSTALLS = False

OPTIONAL_PACKAGES = {
    "autoattack": ("git+https://github.com/fra31/auto-attack.git", True),
}

# Offline Kaggle support: attach a dataset named `offline-src` containing:
# - autoattack-minimal-kaggle.zip
# - cifar10-offline-kaggle.zip
from pathlib import Path
import os
import sys
import tarfile
import zipfile

OFFLINE_SRC_CANDIDATES = [
    Path("/kaggle/input/offline-src"),
    Path("/kaggle/input/datasets/ihatethesis/offline-src"),
]
OFFLINE_SRC_ROOT = next((path for path in OFFLINE_SRC_CANDIDATES if path.exists()), None)

if OFFLINE_SRC_ROOT is not None:
    print(f"Using offline source dataset: {OFFLINE_SRC_ROOT}")
    autoattack_candidates = list(OFFLINE_SRC_ROOT.rglob("autoattack-minimal-kaggle.zip"))
    autoattack_package_candidates = list(OFFLINE_SRC_ROOT.rglob("autoattack/__init__.py"))
    if autoattack_package_candidates:
        autoattack_package_parent = autoattack_package_candidates[0].parent.parent
        sys.path.insert(0, str(autoattack_package_parent))
        try:
            from autoattack import AutoAttack
            print(f"AutoAttack import OK from extracted offline dataset: {autoattack_package_parent}")
        except Exception as exc:
            print(f"AutoAttack extracted import failed: {exc!r}")
    elif autoattack_candidates:
        autoattack_zip = autoattack_candidates[0]
        autoattack_dir = Path("/kaggle/working/autoattack-minimal")
        if not autoattack_dir.exists():
            with zipfile.ZipFile(autoattack_zip, "r") as zf:
                zf.extractall(autoattack_dir)
        sys.path.insert(0, str(autoattack_dir))
        try:
            from autoattack import AutoAttack
            print(f"AutoAttack import OK from offline dataset: {autoattack_zip}")
        except Exception as exc:
            print(f"AutoAttack offline import failed: {exc!r}")
    else:
        print("offline-src found, but no AutoAttack zip or extracted autoattack package was found.")

    cifar_candidates = list(OFFLINE_SRC_ROOT.rglob("cifar10-offline-kaggle.zip"))
    cifar_batch_candidates = list(OFFLINE_SRC_ROOT.rglob("cifar-10-batches-py"))
    if cifar_batch_candidates:
        cifar_batch_dir = cifar_batch_candidates[0]
        data_root = cifar_batch_dir.parent
        os.environ["VT_OFFLINE_CIFAR_ROOT"] = str(data_root)
        print(f"CIFAR-10 offline data found at: {data_root}")
    elif cifar_candidates:
        cifar_zip = cifar_candidates[0]
        cifar_work = Path("/kaggle/working/cifar10-offline")
        data_root = Path("/kaggle/working/data")
        if not cifar_work.exists():
            with zipfile.ZipFile(cifar_zip, "r") as zf:
                zf.extractall(cifar_work)
        tar_path = cifar_work / "cifar-10-python.tar.gz"
        if tar_path.exists() and not (data_root / "cifar-10-batches-py").exists():
            data_root.mkdir(exist_ok=True)
            with tarfile.open(tar_path, "r:gz") as tar:
                tar.extractall(data_root)
        os.environ["VT_OFFLINE_CIFAR_ROOT"] = str(data_root)
        print(f"CIFAR-10 offline data ready at: {data_root}")
    else:
        print("offline-src found, but no CIFAR-10 zip or extracted cifar-10-batches-py folder was found.")
else:
    print("offline-src dataset not attached/found; falling back to existing Kaggle/working paths.")

if RUN_OPTIONAL_INSTALLS:
    import subprocess
    import sys

    for module_name, install_spec in OPTIONAL_PACKAGES.items():
        try:
            __import__(module_name)
        except ImportError:
            install_target, no_deps = install_spec
            command = [sys.executable, "-m", "pip", "install", "-q", install_target]
            if no_deps:
                command.append("--no-deps")
            print(f"Installing {install_target} ...")
            subprocess.check_call(command)


## Configs

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, replace
from pathlib import Path

import torch


def _default_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


@dataclass(frozen=True)
class DatasetConfig:
    name: str = "cifar10"
    root: str = "./data"
    batch_size: int = 128
    num_workers: int = 2
    pin_memory: bool = True
    train_subset: int | None = None
    val_subset: int | None = None
    calib_subset: int | None = None
    test_subset: int | None = None
    augment: bool = True
    event_bins: int = 8
    event_resize: int = 32
    seed: int = 0
    val_fraction: float = 0.1
    calibration_fraction: float = 0.1


@dataclass(frozen=True)
class ModelConfig:
    architecture: str = "resnet"
    num_classes: int = 10
    in_channels: int = 3
    max_time_steps: int = 8
    input_encoding: str = "direct"
    stem_channels: int = 128
    stage_channels: tuple[int, ...] = (128, 256, 512)
    blocks_per_stage: tuple[int, ...] = (3, 3, 2)
    widen_factor: int = 10
    wrn_depth: int = 28
    dropout_rate: float = 0.0
    vgg_classifier_dims: tuple[int, ...] = (4096, 4096)
    stochastic: bool = True
    learnable_sigma: bool = True
    use_dynamic_sigma: bool = True
    use_shortcut_lif: bool = True
    v_threshold: float = 0.5
    lif_tau: float = 2.0
    detach_reset: bool = True
    sigma_init: float = 0.55
    sigma_init_space: str = "raw"
    fixed_sigma_raw: float = -10.0
    normalization_mean: tuple[float, ...] | None = (0.4914, 0.4822, 0.4465)
    normalization_std: tuple[float, ...] | None = (0.2471, 0.2435, 0.2616)


@dataclass(frozen=True)
class TrainingConfig:
    epochs: int = 100
    base_lr: float = 0.125
    sigma_lr_scale: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 0.0
    warmup_epochs: int = 5
    use_amp: bool = True
    amp_dtype: str = "float16"
    gradient_clip: float = 1.0
    clean_ratio_start: float = 0.5
    clean_ratio_end: float = 0.1
    entropy_lambda: float = 0.01
    epsilon: float = 4 / 255
    alpha: float = 4 / 255
    random_start_scale: float = 0.1
    use_parseval: bool = True
    parseval_beta: float = 0.004
    parseval_interval: int = 5
    mixed_time_steps: tuple[int, ...] = (8,)
    seed: int = 0
    device: str = field(default_factory=_default_device)
    output_dir: str = "vt_research_suite/outputs"
    checkpoint_name: str = "model.pt"
    eval_interval: int = 1
    selection_metric: str = "clean"


@dataclass(frozen=True)
class AttackConfig:
    attack_name: str = "pgd"
    epsilon: float = 8 / 255
    alpha: float = 2 / 255
    steps: int = 10
    random_start: bool = True
    eot_iters: int = 4
    exit_aware: bool = False
    objective: str = "untargeted"
    wrong_exit_weight: float = 0.25
    compute_weight: float = 1.0
    preserve_label_weight: float = 0.5
    force_full_trace: bool = True
    attack_time_steps: int | None = None
    autoattack_version: str | None = None
    autoattack_seed: int | None = 0
    autoattack_verbose: bool = False


@dataclass(frozen=True)
class EvaluationConfig:
    vt_thresholds: tuple[float, ...] = (0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99)
    fixed_time_steps: tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7, 8)
    stopping_rules: tuple[str, ...] = (
        "max_softmax",
        "margin",
        "entropy",
        "temperature_softmax",
        "evidential_surrogate",
    )
    seeds: tuple[int, ...] = (0, 1, 2)
    max_batches: int | None = None
    compute_metrics: bool = True
    exact_vt_compute: bool = True
    match_metric: str = "avg_synops"


@dataclass(frozen=True)
class AblationSpec:
    name: str
    vt_enabled: bool = True
    learnable_sigma: bool = False
    use_dynamic_sigma: bool = False
    use_parseval: bool = False
    use_mixed_time_training: bool = False


def default_ablation_specs(max_time_steps: int) -> list[AblationSpec]:
    return [
        AblationSpec("baseline_robust_snn", False, False, False, False, False),
        AblationSpec("baseline_plus_vt", True, False, False, False, False),
        AblationSpec("baseline_plus_vt_plus_dynamic_sigma", True, True, True, False, False),
        AblationSpec("baseline_plus_vt_plus_parseval", True, False, False, True, False),
        AblationSpec("full_model", True, True, True, True, True),
    ]


def apply_ablation(
    model_config: ModelConfig,
    training_config: TrainingConfig,
    spec: AblationSpec,
) -> tuple[ModelConfig, TrainingConfig]:
    mixed_steps = (
        tuple(range(1, model_config.max_time_steps + 1))
        if spec.use_mixed_time_training
        else (model_config.max_time_steps,)
    )
    model_variant = replace(
        model_config,
        learnable_sigma=spec.learnable_sigma,
        use_dynamic_sigma=spec.use_dynamic_sigma,
    )
    training_variant = replace(
        training_config,
        use_parseval=spec.use_parseval,
        mixed_time_steps=mixed_steps,
        checkpoint_name=f"{spec.name}.pt",
    )
    return model_variant, training_variant


def ensure_output_dir(training_config: TrainingConfig) -> Path:
    path = Path(training_config.output_dir)
    path.mkdir(parents=True, exist_ok=True)
    return path


## Data

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms


try:
    import tonic
except ImportError:
    tonic = None


@dataclass(frozen=True)
class DatasetMetadata:
    name: str
    num_classes: int
    in_channels: int
    mean: tuple[float, ...] | None = None
    std: tuple[float, ...] | None = None
    is_event: bool = False


@dataclass(frozen=True)
class DataSplitBundle:
    train_loader: DataLoader
    val_loader: DataLoader
    calib_loader: DataLoader
    test_loader: DataLoader
    metadata: DatasetMetadata


def _loader_kwargs(
    loader: DataLoader,
    *,
    dataset: Dataset,
    batch_size: int | None = None,
    shuffle: bool | None = None,
    num_workers: int | None = None,
    pin_memory: bool | None = None,
) -> dict:
    target_batch_size = loader.batch_size if batch_size is None else batch_size
    target_shuffle = False if shuffle is None else shuffle
    target_num_workers = loader.num_workers if num_workers is None else num_workers
    target_pin_memory = loader.pin_memory if pin_memory is None else pin_memory

    kwargs = {
        "dataset": dataset,
        "batch_size": target_batch_size,
        "shuffle": target_shuffle,
        "num_workers": target_num_workers,
        "pin_memory": target_pin_memory,
        "drop_last": loader.drop_last,
        "collate_fn": loader.collate_fn,
        "worker_init_fn": loader.worker_init_fn,
        "generator": loader.generator,
        "persistent_workers": loader.persistent_workers if target_num_workers > 0 else False,
    }

    prefetch_factor = getattr(loader, "prefetch_factor", None)
    if target_num_workers > 0 and prefetch_factor is not None:
        kwargs["prefetch_factor"] = prefetch_factor

    timeout = getattr(loader, "timeout", 0)
    if timeout:
        kwargs["timeout"] = timeout

    multiprocessing_context = getattr(loader, "multiprocessing_context", None)
    if multiprocessing_context is not None:
        kwargs["multiprocessing_context"] = multiprocessing_context

    pin_memory_device = getattr(loader, "pin_memory_device", "")
    if pin_memory_device:
        kwargs["pin_memory_device"] = pin_memory_device
    return kwargs


def _standard_loader_runtime_kwargs(num_workers: int) -> dict:
    kwargs = {
        "persistent_workers": num_workers > 0,
    }
    if num_workers > 0:
        kwargs["prefetch_factor"] = 4
    return kwargs


def rebuild_loader(
    loader: DataLoader,
    batch_size: int | None = None,
    *,
    shuffle: bool | None = None,
    num_workers: int | None = None,
    pin_memory: bool | None = None,
) -> DataLoader:
    return DataLoader(
        **_loader_kwargs(
            loader,
            dataset=loader.dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )
    )


def _extract_targets(dataset: Dataset) -> list[int]:
    if isinstance(dataset, Subset):
        parent_targets = _extract_targets(dataset.dataset)
        return [int(parent_targets[index]) for index in dataset.indices]

    for attr in ("targets", "labels"):
        if hasattr(dataset, attr):
            values = getattr(dataset, attr)
            if isinstance(values, torch.Tensor):
                return [int(value) for value in values.tolist()]
            return [int(value) for value in list(values)]

    raise ValueError(
        f"Could not extract class targets from dataset type {type(dataset).__name__}. "
        "Balanced attack subsets currently require datasets exposing `targets` or `labels`."
    )


def build_subset_loader(
    loader: DataLoader,
    subset_size: int,
    seed: int,
    *,
    balanced: bool = False,
    batch_size: int | None = None,
    shuffle: bool = False,
) -> DataLoader:
    if subset_size <= 0:
        raise ValueError("subset_size must be positive.")

    base_dataset = loader.dataset
    if subset_size >= len(base_dataset):
        return rebuild_loader(loader, batch_size=batch_size, shuffle=shuffle)

    generator = torch.Generator().manual_seed(seed)

    if balanced:
        targets = _extract_targets(base_dataset)
        class_values = sorted(set(targets))
        if not class_values:
            raise ValueError("Balanced subset construction requires at least one class label.")
        per_class = subset_size // len(class_values)
        remainder = subset_size % len(class_values)
        if per_class <= 0:
            raise ValueError(
                f"subset_size={subset_size} is too small for {len(class_values)} classes."
            )

        selected_indices: list[int] = []
        for class_rank, class_value in enumerate(class_values):
            class_indices = [idx for idx, target in enumerate(targets) if target == class_value]
            target_count = per_class + (1 if class_rank < remainder else 0)
            if len(class_indices) < target_count:
                raise ValueError(
                    f"Class {class_value} only has {len(class_indices)} samples, but {target_count} were requested."
                )
            order = torch.randperm(len(class_indices), generator=generator).tolist()
            selected_indices.extend(class_indices[idx] for idx in order[:target_count])
        shuffle_order = torch.randperm(len(selected_indices), generator=generator).tolist()
        selected_indices = [selected_indices[idx] for idx in shuffle_order]
        subset_dataset = Subset(base_dataset, selected_indices)
    else:
        subset_dataset = _maybe_subset(base_dataset, subset_size, seed)

    return DataLoader(
        **_loader_kwargs(
            loader,
            dataset=subset_dataset,
            batch_size=batch_size,
            shuffle=shuffle,
        )
    )


class Cutout:
    def __init__(self, n_holes: int = 1, length: int = 16):
        self.n_holes = n_holes
        self.length = length

    def __call__(self, image: torch.Tensor) -> torch.Tensor:
        height, width = image.size(1), image.size(2)
        mask = np.ones((height, width), dtype=np.float32)
        for _ in range(self.n_holes):
            y = np.random.randint(height)
            x = np.random.randint(width)
            y1 = np.clip(y - self.length // 2, 0, height)
            y2 = np.clip(y + self.length // 2, 0, height)
            x1 = np.clip(x - self.length // 2, 0, width)
            x2 = np.clip(x + self.length // 2, 0, width)
            mask[y1:y2, x1:x2] = 0.0
        return image * torch.from_numpy(mask).to(image.dtype).expand_as(image)


class EventFramesToTensor:
    def __init__(self, resize: int = 32):
        self.resize = resize

    def __call__(self, frames: np.ndarray | torch.Tensor) -> torch.Tensor:
        tensor = torch.as_tensor(frames, dtype=torch.float32)
        if tensor.ndim == 3:
            tensor = tensor.unsqueeze(1)
        tensor = F.interpolate(
            tensor,
            size=(self.resize, self.resize),
            mode="bilinear",
            align_corners=False,
        )
        scale = tensor.amax(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        return tensor / scale


_DATASET_REGISTRY: dict[str, DatasetMetadata] = {
    "cifar10": DatasetMetadata(
        name="cifar10",
        num_classes=10,
        in_channels=3,
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2471, 0.2435, 0.2616),
    ),
    "cifar100": DatasetMetadata(
        name="cifar100",
        num_classes=100,
        in_channels=3,
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761),
    ),
    "mnist_dvs": DatasetMetadata("mnist_dvs", 10, 2, is_event=True),
    "dvs128_gesture": DatasetMetadata("dvs128_gesture", 11, 2, is_event=True),
}


def get_dataset_metadata(name: str) -> DatasetMetadata:
    key = name.lower()
    if key not in _DATASET_REGISTRY:
        raise ValueError(f"Unsupported dataset: {name}")
    return _DATASET_REGISTRY[key]


def _maybe_subset(dataset: Dataset, subset_size: int | None, seed: int) -> Dataset:
    if subset_size is None or subset_size >= len(dataset):
        return dataset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:subset_size].tolist()
    return Subset(dataset, indices)


def _split_indices(length: int, val_fraction: float, calibration_fraction: float, seed: int) -> tuple[list[int], list[int], list[int]]:
    if val_fraction < 0 or calibration_fraction < 0 or val_fraction + calibration_fraction >= 1.0:
        raise ValueError("val_fraction and calibration_fraction must be non-negative and sum to less than 1.")
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(length, generator=generator).tolist()
    val_count = int(length * val_fraction)
    calib_count = int(length * calibration_fraction)
    train_count = length - val_count - calib_count
    train_indices = indices[:train_count]
    val_indices = indices[train_count:train_count + val_count]
    calib_indices = indices[train_count + val_count:]
    return train_indices, val_indices, calib_indices


def _build_cifar_datasets(config: DatasetConfig) -> tuple[Dataset, Dataset]:
    if config.augment:
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                Cutout(n_holes=1, length=16),
            ]
        )
    else:
        train_transform = transforms.Compose([transforms.ToTensor()])
    test_transform = transforms.Compose([transforms.ToTensor()])

    if config.name == "cifar10":
        train_dataset = datasets.CIFAR10(config.root, train=True, download=True, transform=train_transform)
        test_dataset = datasets.CIFAR10(config.root, train=False, download=True, transform=test_transform)
    else:
        train_dataset = datasets.CIFAR100(config.root, train=True, download=True, transform=train_transform)
        test_dataset = datasets.CIFAR100(config.root, train=False, download=True, transform=test_transform)
    return train_dataset, test_dataset


def _build_cifar_split_datasets(config: DatasetConfig) -> tuple[Dataset, Dataset, Dataset]:
    if config.augment:
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                Cutout(n_holes=1, length=16),
            ]
        )
    else:
        train_transform = transforms.Compose([transforms.ToTensor()])
    eval_transform = transforms.Compose([transforms.ToTensor()])

    dataset_cls = datasets.CIFAR10 if config.name == "cifar10" else datasets.CIFAR100
    train_aug_dataset = dataset_cls(config.root, train=True, download=True, transform=train_transform)
    train_eval_dataset = dataset_cls(config.root, train=True, download=True, transform=eval_transform)
    test_dataset = dataset_cls(config.root, train=False, download=True, transform=eval_transform)
    return train_aug_dataset, train_eval_dataset, test_dataset


def _build_event_datasets(config: DatasetConfig) -> tuple[Dataset, Dataset]:
    if tonic is None:
        raise ImportError(
            "Event datasets require `tonic`. Install it before running MNIST-DVS or DVS128 Gesture experiments."
        )

    if config.name == "mnist_dvs":
        dataset_cls = tonic.datasets.MNISTDVS
    else:
        dataset_cls = tonic.datasets.DVSGesture

    frame_transform = tonic.transforms.Compose(
        [
            tonic.transforms.ToFrame(sensor_size=dataset_cls.sensor_size, n_time_bins=config.event_bins),
            EventFramesToTensor(resize=config.event_resize),
        ]
    )
    train_dataset = dataset_cls(save_to=config.root, train=True, transform=frame_transform)
    test_dataset = dataset_cls(save_to=config.root, train=False, transform=frame_transform)
    return train_dataset, test_dataset


def build_dataloaders(config: DatasetConfig) -> tuple[DataLoader, DataLoader, DatasetMetadata]:
    metadata = get_dataset_metadata(config.name)
    if metadata.is_event:
        train_dataset, test_dataset = _build_event_datasets(config)
    else:
        train_dataset, test_dataset = _build_cifar_datasets(config)

    train_dataset = _maybe_subset(train_dataset, config.train_subset, config.seed)
    test_dataset = _maybe_subset(test_dataset, config.test_subset, config.seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        drop_last=not metadata.is_event,
        **_standard_loader_runtime_kwargs(config.num_workers),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        drop_last=False,
        **_standard_loader_runtime_kwargs(config.num_workers),
    )
    return train_loader, test_loader, metadata


def build_data_splits(config: DatasetConfig) -> DataSplitBundle:
    metadata = get_dataset_metadata(config.name)

    if metadata.is_event:
        train_dataset, test_dataset = _build_event_datasets(config)
        train_eval_dataset = train_dataset
    else:
        train_dataset, train_eval_dataset, test_dataset = _build_cifar_split_datasets(config)

    train_indices, val_indices, calib_indices = _split_indices(
        len(train_dataset),
        config.val_fraction,
        config.calibration_fraction,
        config.seed,
    )

    val_subset_size = config.val_subset if config.val_subset is not None else config.test_subset
    calib_subset_size = config.calib_subset if config.calib_subset is not None else config.test_subset

    train_subset = _maybe_subset(Subset(train_dataset, train_indices), config.train_subset, config.seed)
    val_subset = _maybe_subset(Subset(train_eval_dataset, val_indices), val_subset_size, config.seed)
    calib_subset = _maybe_subset(Subset(train_eval_dataset, calib_indices), calib_subset_size, config.seed)
    test_subset = _maybe_subset(test_dataset, config.test_subset, config.seed)

    train_loader = DataLoader(
        train_subset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        drop_last=not metadata.is_event,
        **_standard_loader_runtime_kwargs(config.num_workers),
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        drop_last=False,
        **_standard_loader_runtime_kwargs(config.num_workers),
    )
    calib_loader = DataLoader(
        calib_subset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        drop_last=False,
        **_standard_loader_runtime_kwargs(config.num_workers),
    )
    test_loader = DataLoader(
        test_subset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        drop_last=False,
        **_standard_loader_runtime_kwargs(config.num_workers),
    )

    return DataSplitBundle(
        train_loader=train_loader,
        val_loader=val_loader,
        calib_loader=calib_loader,
        test_loader=test_loader,
        metadata=metadata,
    )


## Stopping

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn.functional as F


@dataclass
class StopRule:
    name: str

    def confidence(self, logits: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def should_stop(self, logits: torch.Tensor, threshold: float) -> torch.Tensor:
        return self.confidence(logits) >= threshold


class MaxSoftmaxStopRule(StopRule):
    def __init__(self) -> None:
        super().__init__(name="max_softmax")

    def confidence(self, logits: torch.Tensor) -> torch.Tensor:
        return F.softmax(logits, dim=-1).amax(dim=-1)


class MarginStopRule(StopRule):
    def __init__(self) -> None:
        super().__init__(name="margin")

    def confidence(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        top2 = torch.topk(probs, k=2, dim=-1).values
        return (top2[..., 0] - top2[..., 1]).clamp(0.0, 1.0)


class EntropyStopRule(StopRule):
    def __init__(self) -> None:
        super().__init__(name="entropy")

    def confidence(self, logits: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=-1)
        entropy = -(probs * probs.clamp_min(1e-8).log()).sum(dim=-1)
        return (1.0 - entropy / math.log(logits.size(-1))).clamp(0.0, 1.0)


class TemperatureScaledStopRule(StopRule):
    def __init__(self, temperature: float = 2.0) -> None:
        super().__init__(name="temperature_softmax")
        self.temperature = temperature

    def confidence(self, logits: torch.Tensor) -> torch.Tensor:
        return F.softmax(logits / self.temperature, dim=-1).amax(dim=-1)


class EvidentialStopRule(StopRule):
    def __init__(self) -> None:
        super().__init__(name="evidential_surrogate")

    def confidence(self, logits: torch.Tensor) -> torch.Tensor:
        alpha = F.softplus(logits) + 1.0
        uncertainty = logits.size(-1) / alpha.sum(dim=-1)
        return (1.0 - uncertainty).clamp(0.0, 1.0)


def build_stop_rule(name: str, **kwargs) -> StopRule:
    key = name.lower()
    if key == "max_softmax":
        return MaxSoftmaxStopRule()
    if key == "margin":
        return MarginStopRule()
    if key == "entropy":
        return EntropyStopRule()
    if key in {"temperature", "temperature_softmax"}:
        return TemperatureScaledStopRule(temperature=kwargs.get("temperature", 2.0))
    if key in {"evidential", "evidential_surrogate"}:
        return EvidentialStopRule()
    raise ValueError(f"Unsupported stop rule: {name}")


## Model

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F



def _softplus_inverse(value: float) -> float:
    return math.log(math.exp(value) - 1.0)


class SurrogateBPFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, inputs: torch.Tensor, alpha: float = 4.0) -> torch.Tensor:
        ctx.save_for_backward(inputs)
        ctx.alpha = alpha
        return (inputs >= 0).to(inputs.dtype)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor) -> tuple[torch.Tensor, None]:
        (inputs,) = ctx.saved_tensors
        alpha = ctx.alpha
        grad_input = grad_output * alpha / (2 * (1 + (alpha * inputs).abs()) ** 2)
        return grad_input, None


class NormalizeByChannelMeanStd(nn.Module):
    def __init__(self, mean: tuple[float, ...], std: tuple[float, ...]) -> None:
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, -1, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, -1, 1, 1))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        if inputs.ndim == 4:
            return (inputs - self.mean) / self.std
        if inputs.ndim == 5:
            batch, steps, channels, height, width = inputs.shape
            flat = inputs.view(batch * steps, channels, height, width)
            flat = (flat - self.mean) / self.std
            return flat.view(batch, steps, channels, height, width)
        raise ValueError(f"Unexpected input shape: {tuple(inputs.shape)}")


class SENoiseScaler(nn.Module):
    def __init__(self, channels: int, reduction: int = 4, base_scale: float = 1.0) -> None:
        super().__init__()
        reduced = max(channels // reduction, 8)
        self.base_scale = base_scale
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(
            nn.Linear(channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, channels, bias=False),
            nn.Sigmoid(),
        )
        self.scale_factor = nn.Parameter(torch.tensor(0.5))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch, channels, _, _ = inputs.shape
        squeezed = self.squeeze(inputs).view(batch, channels)
        excited = self.excite(squeezed).view(batch, channels, 1, 1)
        scale = self.base_scale + F.softplus(self.scale_factor) * excited
        return scale.clamp(0.5, 2.5)


class AdaptiveRHLIFNode(nn.Module):
    is_spiking_neuron = True

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.v_threshold = config.v_threshold
        self.stochastic = config.stochastic
        self.learnable_sigma = config.learnable_sigma and self.stochastic
        if self.learnable_sigma:
            if config.sigma_init_space == "raw":
                raw_sigma = config.sigma_init
            elif config.sigma_init_space == "sigma":
                raw_sigma = _softplus_inverse(config.sigma_init)
            else:
                raise ValueError(f"Unsupported sigma_init_space: {config.sigma_init_space}")
        else:
            raw_sigma = config.fixed_sigma_raw
        if self.learnable_sigma:
            self.sigma_raw = nn.Parameter(torch.tensor(raw_sigma, dtype=torch.float32))
        else:
            self.register_buffer("sigma_raw", torch.tensor(raw_sigma, dtype=torch.float32))
        self.v: torch.Tensor | None = None

    @property
    def sigma(self) -> torch.Tensor:
        return F.softplus(self.sigma_raw)

    def forward(self, inputs: torch.Tensor, sigma_scaler: torch.Tensor | None = None) -> torch.Tensor:
        if self.v is None or self.v.shape != inputs.shape:
            self.v = torch.zeros_like(inputs)
        if self.stochastic:
            final_sigma = self.sigma if sigma_scaler is None else self.sigma * sigma_scaler
            noise = torch.randn_like(inputs) * final_sigma
            gate = torch.sigmoid(noise)
        else:
            # Deterministic control: same membrane update structure, but without
            # stochastic noise injection. Since the stochastic gate has zero-mean
            # logits, its expected value is 0.5; we use that fixed leak factor here.
            gate = torch.full_like(inputs, 0.5)
        self.v = gate * self.v + (1 - gate) * inputs
        spike = SurrogateBPFunction.apply(self.v - self.v_threshold)
        self.v = self.v * (1 - spike)
        return spike

    def reset(self) -> None:
        self.v = None


class VanillaLIFNode(nn.Module):
    is_spiking_neuron = True

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.v_threshold = config.v_threshold
        self.tau = config.lif_tau
        self.detach_reset = config.detach_reset
        self.v: torch.Tensor | None = None

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        if self.v is None or self.v.shape != inputs.shape:
            self.v = torch.zeros_like(inputs)

        # Standard leaky integrate-and-fire update used for the vanilla deterministic path.
        self.v = self.v + (inputs - self.v) / self.tau
        spike = SurrogateBPFunction.apply(self.v - self.v_threshold)
        reset_spike = spike.detach() if self.detach_reset else spike
        self.v = self.v * (1 - reset_spike)
        return spike

    def reset(self) -> None:
        self.v = None


class AdaptiveConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, config: ModelConfig, stride: int = 1) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.scaler = SENoiseScaler(out_channels) if config.use_dynamic_sigma else None
        self.lif = AdaptiveRHLIFNode(config)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.bn(self.conv(inputs))
        return self.lif(out, sigma_scaler=self.scaler(out) if self.scaler is not None else None)


class AdaptiveLinearBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int, config: ModelConfig) -> None:
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.lif = AdaptiveRHLIFNode(config)
        self.dropout = nn.Dropout(config.dropout_rate) if config.dropout_rate > 0 else nn.Identity()

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.linear(inputs)
        out = self.lif(out)
        return self.dropout(out)


class VanillaConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, config: ModelConfig, stride: int = 1) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.lif = VanillaLIFNode(config)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.lif(self.bn(self.conv(inputs)))


class VanillaLinearBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int, config: ModelConfig) -> None:
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.lif = VanillaLIFNode(config)
        self.dropout = nn.Dropout(config.dropout_rate) if config.dropout_rate > 0 else nn.Identity()

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.linear(inputs)
        out = self.lif(out)
        return self.dropout(out)


class AdaptiveResidualBlock(nn.Module):
    def __init__(self, in_planes: int, planes: int, stride: int, config: ModelConfig) -> None:
        super().__init__()
        self.has_shortcut = stride != 1 or in_planes != planes

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.attn1 = SENoiseScaler(planes) if config.use_dynamic_sigma else None
        self.lif1 = AdaptiveRHLIFNode(config)

        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.attn2 = SENoiseScaler(planes) if config.use_dynamic_sigma else None
        self.lif2 = AdaptiveRHLIFNode(config)

        if self.has_shortcut:
            self.shortcut_conv = nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False)
            self.shortcut_bn = nn.BatchNorm2d(planes)
            self.shortcut_attn = SENoiseScaler(planes) if config.use_dynamic_sigma else None
            self.shortcut_lif = AdaptiveRHLIFNode(config) if config.use_shortcut_lif else None
        else:
            self.shortcut_conv = None
            self.shortcut_bn = None
            self.shortcut_attn = None
            self.shortcut_lif = None

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.bn1(self.conv1(inputs))
        out = self.lif1(out, sigma_scaler=self.attn1(out) if self.attn1 is not None else None)

        out = self.bn2(self.conv2(out))
        out = self.lif2(out, sigma_scaler=self.attn2(out) if self.attn2 is not None else None)

        if self.has_shortcut:
            shortcut = self.shortcut_bn(self.shortcut_conv(inputs))
            if self.shortcut_lif is not None:
                shortcut = self.shortcut_lif(
                    shortcut,
                    sigma_scaler=self.shortcut_attn(shortcut) if self.shortcut_attn is not None else None,
                )
        else:
            shortcut = inputs
        return out + shortcut


class AdaptiveWideResidualBlock(nn.Module):
    def __init__(self, in_planes: int, planes: int, stride: int, config: ModelConfig) -> None:
        super().__init__()
        self.norm1 = nn.BatchNorm2d(in_planes)
        self.scaler1 = SENoiseScaler(in_planes) if config.use_dynamic_sigma else None
        self.lif1 = AdaptiveRHLIFNode(config)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)

        self.norm2 = nn.BatchNorm2d(planes)
        self.scaler2 = SENoiseScaler(planes) if config.use_dynamic_sigma else None
        self.lif2 = AdaptiveRHLIFNode(config)
        self.dropout = nn.Dropout(config.dropout_rate) if config.dropout_rate > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)

        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.norm1(inputs)
        out = self.lif1(out, sigma_scaler=self.scaler1(out) if self.scaler1 is not None else None)
        shortcut = self.shortcut(out) if self.shortcut is not None else inputs
        out = self.conv1(out)

        out = self.norm2(out)
        out = self.lif2(out, sigma_scaler=self.scaler2(out) if self.scaler2 is not None else None)
        out = self.dropout(out)
        out = self.conv2(out)
        return out + shortcut


class VanillaResidualBlock(nn.Module):
    def __init__(self, in_planes: int, planes: int, stride: int, config: ModelConfig) -> None:
        super().__init__()
        self.has_shortcut = stride != 1 or in_planes != planes

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.lif1 = VanillaLIFNode(config)

        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.lif2 = VanillaLIFNode(config)

        if self.has_shortcut:
            self.shortcut_conv = nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False)
            self.shortcut_bn = nn.BatchNorm2d(planes)
            self.shortcut_lif = VanillaLIFNode(config) if config.use_shortcut_lif else None
        else:
            self.shortcut_conv = None
            self.shortcut_bn = None
            self.shortcut_lif = None

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.lif1(self.bn1(self.conv1(inputs)))
        out = self.lif2(self.bn2(self.conv2(out)))

        if self.has_shortcut:
            shortcut = self.shortcut_bn(self.shortcut_conv(inputs))
            if self.shortcut_lif is not None:
                shortcut = self.shortcut_lif(shortcut)
        else:
            shortcut = inputs

        return out + shortcut


class VanillaInvertedResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int, expand_ratio: int, config: ModelConfig) -> None:
        super().__init__()
        if stride not in (1, 2):
            raise ValueError("MobileNetV2 inverted residual stride must be 1 or 2.")

        hidden_dim = int(round(in_channels * expand_ratio))
        self.use_residual = stride == 1 and in_channels == out_channels
        layers: list[nn.Module] = []

        if expand_ratio != 1:
            layers.extend(
                [
                    nn.Conv2d(in_channels, hidden_dim, kernel_size=1, stride=1, padding=0, bias=False),
                    nn.BatchNorm2d(hidden_dim),
                    VanillaLIFNode(config),
                ]
            )

        layers.extend(
            [
                nn.Conv2d(
                    hidden_dim,
                    hidden_dim,
                    kernel_size=3,
                    stride=stride,
                    padding=1,
                    groups=hidden_dim,
                    bias=False,
                ),
                nn.BatchNorm2d(hidden_dim),
                VanillaLIFNode(config),
                nn.Conv2d(hidden_dim, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_channels),
            ]
        )
        self.block = nn.Sequential(*layers)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.block(inputs)
        if self.use_residual:
            out = out + inputs
        return out


@dataclass
class TraceResult:
    final_logits: torch.Tensor
    steps_taken: torch.Tensor
    avg_logits_per_step: torch.Tensor
    raw_logits_per_step: torch.Tensor
    executed_steps: int
    confidence_per_step: torch.Tensor | None = None
    stop_mask_per_step: torch.Tensor | None = None


class TraceableSpikingModel(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.normalize = (
            NormalizeByChannelMeanStd(config.normalization_mean, config.normalization_std)
            if config.normalization_mean is not None and config.normalization_std is not None
            else None
        )

    def reset(self) -> None:
        for module in self.modules():
            if hasattr(module, "reset") and module is not self:
                module.reset()

    def _prepare_inputs(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.normalize(inputs) if self.normalize is not None else inputs

    def _step_input(self, inputs: torch.Tensor, step: int) -> torch.Tensor:
        if inputs.ndim == 5:
            return inputs[:, step]
        if self.config.input_encoding.lower() == "direct":
            return inputs
        raise ValueError(f"Unsupported static-input encoding: {self.config.input_encoding}")

    def _forward_one_step(self, inputs: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def forward_trace(
        self,
        inputs: torch.Tensor,
        time_steps: int | None = None,
        stop_rule: StopRule | None = None,
        threshold: float = 0.95,
        force_full: bool = False,
    ) -> TraceResult:
        prepared = self._prepare_inputs(inputs)
        max_steps = time_steps or self.config.max_time_steps
        if prepared.ndim == 5:
            max_steps = min(max_steps, prepared.size(1))

        self.reset()
        batch_size = prepared.size(0)
        device = prepared.device
        accumulated = torch.zeros(batch_size, self.config.num_classes, device=device, dtype=prepared.dtype)
        raw_logits = []
        avg_logits = []
        confidences = []
        stop_masks = []
        final_logits = torch.zeros(batch_size, self.config.num_classes, device=device, dtype=prepared.dtype)
        steps_taken = torch.full((batch_size,), max_steps, device=device, dtype=torch.long)
        finished = torch.zeros(batch_size, device=device, dtype=torch.bool)

        for step in range(max_steps):
            logits = self._forward_one_step(self._step_input(prepared, step))
            accumulated = accumulated + logits
            averaged = accumulated / float(step + 1)
            raw_logits.append(logits)
            avg_logits.append(averaged)

            if stop_rule is not None:
                confidence = stop_rule.confidence(averaged)
                should_stop = stop_rule.should_stop(averaged, threshold)
                newly_finished = should_stop & (~finished)
                if newly_finished.any():
                    final_logits[newly_finished] = averaged[newly_finished]
                    steps_taken[newly_finished] = step + 1
                finished = finished | newly_finished
                confidences.append(confidence)
                stop_masks.append(should_stop)
                if finished.all() and not force_full:
                    break

        raw_logits_tensor = torch.stack(raw_logits, dim=0)
        avg_logits_tensor = torch.stack(avg_logits, dim=0)
        executed_steps = avg_logits_tensor.size(0)

        if stop_rule is None:
            final = avg_logits_tensor[-1]
            return TraceResult(
                final_logits=final,
                steps_taken=torch.full((batch_size,), executed_steps, device=device, dtype=torch.long),
                avg_logits_per_step=avg_logits_tensor,
                raw_logits_per_step=raw_logits_tensor,
                executed_steps=executed_steps,
            )

        if (~finished).any():
            final_logits[~finished] = avg_logits_tensor[-1][~finished]

        return TraceResult(
            final_logits=final_logits,
            steps_taken=steps_taken,
            avg_logits_per_step=avg_logits_tensor,
            raw_logits_per_step=raw_logits_tensor,
            executed_steps=executed_steps,
            confidence_per_step=torch.stack(confidences, dim=0),
            stop_mask_per_step=torch.stack(stop_masks, dim=0),
        )

    def forward(
        self,
        inputs: torch.Tensor,
        time_steps: int | None = None,
        use_vt: bool = False,
        stop_rule: StopRule | None = None,
        threshold: float = 0.95,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        trace = self.forward_trace(
            inputs,
            time_steps=time_steps,
            stop_rule=stop_rule if use_vt else None,
            threshold=threshold,
        )
        if use_vt:
            return trace.final_logits, trace.steps_taken
        return trace.final_logits


class VTResNetSNN(TraceableSpikingModel):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__(config)
        self.conv1 = nn.Conv2d(
            config.in_channels,
            config.stem_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(config.stem_channels)
        self.lif1 = AdaptiveRHLIFNode(config)
        c1, c2, c3 = config.stage_channels
        b1, b2, b3 = config.blocks_per_stage
        self.layer1 = self._make_layer(config.stem_channels, c1, b1, 1)
        self.layer2 = self._make_layer(c1, c2, b2, 2)
        self.layer3 = self._make_layer(c2, c3, b3, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(c3, config.num_classes)
        self._initialize_weights()

    def _make_layer(self, in_planes: int, planes: int, blocks: int, stride: int) -> nn.Sequential:
        layers = []
        current = in_planes
        for current_stride in [stride] + [1] * (blocks - 1):
            layers.append(AdaptiveResidualBlock(current, planes, current_stride, self.config))
            current = planes
        return nn.Sequential(*layers)

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1.0)
                nn.init.constant_(module.bias, 0.0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0.0, 0.01)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.0)

    def _forward_one_step(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.lif1(self.bn1(self.conv1(inputs)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)


class VTVanillaResNet18SNN(TraceableSpikingModel):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__(config)
        if len(config.stage_channels) != 4 or len(config.blocks_per_stage) != 4:
            raise ValueError(
                "vanilla_resnet18 expects four stages. Use stage_channels=(64, 128, 256, 512) "
                "and blocks_per_stage=(2, 2, 2, 2)."
            )

        self.conv1 = nn.Conv2d(
            config.in_channels,
            config.stem_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(config.stem_channels)
        self.lif1 = VanillaLIFNode(config)

        c1, c2, c3, c4 = config.stage_channels
        b1, b2, b3, b4 = config.blocks_per_stage
        self.layer1 = self._make_layer(config.stem_channels, c1, b1, 1)
        self.layer2 = self._make_layer(c1, c2, b2, 2)
        self.layer3 = self._make_layer(c2, c3, b3, 2)
        self.layer4 = self._make_layer(c3, c4, b4, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(c4, config.num_classes)
        self._initialize_weights()

    def _make_layer(self, in_planes: int, planes: int, blocks: int, stride: int) -> nn.Sequential:
        layers = []
        current = in_planes
        for current_stride in [stride] + [1] * (blocks - 1):
            layers.append(VanillaResidualBlock(current, planes, current_stride, self.config))
            current = planes
        return nn.Sequential(*layers)

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1.0)
                nn.init.constant_(module.bias, 0.0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0.0, 0.01)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.0)

    def _forward_one_step(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.lif1(self.bn1(self.conv1(inputs)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)


class VTVGGSNN(TraceableSpikingModel):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__(config)
        channels = config.stage_channels
        blocks = config.blocks_per_stage
        if len(channels) != len(blocks):
            raise ValueError("VGG stage_channels and blocks_per_stage must have the same length.")
        if len(channels) == 0:
            raise ValueError("VGG requires at least one convolutional stage.")
        current_channels = config.in_channels
        stages = []
        for stage_channels, stage_blocks in zip(channels, blocks):
            layers = []
            for _ in range(stage_blocks):
                layers.append(AdaptiveConvBlock(current_channels, stage_channels, config, stride=1))
                current_channels = stage_channels
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            stages.append(nn.Sequential(*layers))
        self.stages = nn.ModuleList(stages)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        classifier_layers: list[nn.Module] = []
        current_features = current_channels
        for hidden_dim in config.vgg_classifier_dims:
            classifier_layers.append(AdaptiveLinearBlock(current_features, hidden_dim, config))
            current_features = hidden_dim
        classifier_layers.append(nn.Linear(current_features, config.num_classes))
        self.classifier = nn.Sequential(*classifier_layers)
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1.0)
                nn.init.constant_(module.bias, 0.0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0.0, 0.01)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.0)

    def _forward_one_step(self, inputs: torch.Tensor) -> torch.Tensor:
        out = inputs
        for stage in self.stages:
            out = stage(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.classifier(out)


class VTVanillaVGGSNN(TraceableSpikingModel):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__(config)
        channels = config.stage_channels
        blocks = config.blocks_per_stage
        if len(channels) != len(blocks):
            raise ValueError("vanilla_vgg16 stage_channels and blocks_per_stage must have the same length.")
        if len(channels) != 5 or tuple(blocks) != (2, 2, 3, 3, 3):
            raise ValueError(
                "vanilla_vgg16 expects five stages with blocks_per_stage=(2, 2, 3, 3, 3), "
                "e.g. stage_channels=(64, 128, 256, 512, 512)."
            )

        current_channels = config.in_channels
        stages = []
        for stage_channels, stage_blocks in zip(channels, blocks):
            layers = []
            for _ in range(stage_blocks):
                layers.append(VanillaConvBlock(current_channels, stage_channels, config, stride=1))
                current_channels = stage_channels
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            stages.append(nn.Sequential(*layers))
        self.stages = nn.ModuleList(stages)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        classifier_layers: list[nn.Module] = []
        current_features = current_channels
        for hidden_dim in config.vgg_classifier_dims:
            classifier_layers.append(VanillaLinearBlock(current_features, hidden_dim, config))
            current_features = hidden_dim
        classifier_layers.append(nn.Linear(current_features, config.num_classes))
        self.classifier = nn.Sequential(*classifier_layers)
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1.0)
                nn.init.constant_(module.bias, 0.0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0.0, 0.01)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.0)

    def _forward_one_step(self, inputs: torch.Tensor) -> torch.Tensor:
        out = inputs
        for stage in self.stages:
            out = stage(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.classifier(out)


class VTVanillaMobileNetV2SNN(TraceableSpikingModel):
    # MobileNetV2-1.0 block schedule from Sandler et al., adapted for CIFAR.
    # The CIFAR convention keeps the width/depth but avoids the early ImageNet
    # downsampling by using stride 1 in the stem and the 24-channel stage.
    INVERTED_RESIDUAL_SETTINGS: tuple[tuple[int, int, int, int], ...] = (
        (1, 16, 1, 1),
        (6, 24, 2, 1),
        (6, 32, 3, 2),
        (6, 64, 4, 2),
        (6, 96, 3, 1),
        (6, 160, 3, 2),
        (6, 320, 1, 1),
    )

    def __init__(self, config: ModelConfig) -> None:
        super().__init__(config)
        stem_channels = config.stem_channels
        if stem_channels <= 0:
            raise ValueError("vanilla_mobilenetv2 requires a positive stem_channels value.")

        self.stem = nn.Sequential(
            nn.Conv2d(config.in_channels, stem_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            VanillaLIFNode(config),
        )

        layers: list[nn.Module] = []
        input_channels = stem_channels
        for expand_ratio, output_channels, num_blocks, stride in self.INVERTED_RESIDUAL_SETTINGS:
            for block_idx in range(num_blocks):
                block_stride = stride if block_idx == 0 else 1
                layers.append(
                    VanillaInvertedResidualBlock(
                        input_channels,
                        output_channels,
                        block_stride,
                        expand_ratio,
                        config,
                    )
                )
                input_channels = output_channels
        self.features = nn.Sequential(*layers)

        last_channels = 1280
        self.final_conv = nn.Sequential(
            nn.Conv2d(input_channels, last_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(last_channels),
            VanillaLIFNode(config),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(config.dropout_rate) if config.dropout_rate > 0 else nn.Identity()
        self.fc = nn.Linear(last_channels, config.num_classes)
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1.0)
                nn.init.constant_(module.bias, 0.0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0.0, 0.01)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.0)

    def _forward_one_step(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.stem(inputs)
        out = self.features(out)
        out = self.final_conv(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.dropout(out)
        return self.fc(out)


class VTWideResNetSNN(TraceableSpikingModel):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__(config)
        base_width = 16
        if (config.wrn_depth - 4) % 6 != 0 or config.wrn_depth < 10:
            raise ValueError("WRN depth must follow 6n+4 with n >= 1, e.g. 16, 22, 28, 40.")
        depth_blocks = (config.wrn_depth - 4) // 6
        widths = (
            base_width * config.widen_factor,
            2 * base_width * config.widen_factor,
            4 * base_width * config.widen_factor,
        )
        # WRN stage widths and residual block counts are defined by (depth, widen_factor).
        # The generic stage_channels / blocks_per_stage fields are kept for config display
        # and notebook presets, but this backbone follows the canonical WRN construction.

        self.conv1 = nn.Conv2d(config.in_channels, base_width, kernel_size=3, stride=1, padding=1, bias=False)
        self.layer1 = self._make_layer(base_width, widths[0], depth_blocks, stride=1)
        self.layer2 = self._make_layer(widths[0], widths[1], depth_blocks, stride=2)
        self.layer3 = self._make_layer(widths[1], widths[2], depth_blocks, stride=2)
        self.bn = nn.BatchNorm2d(widths[2])
        self.scaler = SENoiseScaler(widths[2]) if config.use_dynamic_sigma else None
        self.lif = AdaptiveRHLIFNode(config)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(widths[2], config.num_classes)
        self._initialize_weights()

    def _make_layer(self, in_planes: int, planes: int, blocks: int, stride: int) -> nn.Sequential:
        layers = []
        current = in_planes
        for current_stride in [stride] + [1] * (blocks - 1):
            layers.append(AdaptiveWideResidualBlock(current, planes, current_stride, self.config))
            current = planes
        return nn.Sequential(*layers)

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1.0)
                nn.init.constant_(module.bias, 0.0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, 0.0, 0.01)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.0)

    def _forward_one_step(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.conv1(inputs)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.bn(out)
        out = self.lif(out, sigma_scaler=self.scaler(out) if self.scaler is not None else None)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)


def build_model(config: ModelConfig, seed: int | None = None) -> TraceableSpikingModel:
    if seed is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    architecture = config.architecture.lower()
    if architecture == "resnet":
        return VTResNetSNN(config)
    if architecture == "vanilla_resnet18":
        return VTVanillaResNet18SNN(config)
    if architecture == "vanilla_vgg16":
        return VTVanillaVGGSNN(config)
    if architecture == "vanilla_mobilenetv2":
        return VTVanillaMobileNetV2SNN(config)
    if architecture == "vgg":
        return VTVGGSNN(config)
    if architecture == "wrn":
        return VTWideResNetSNN(config)
    raise ValueError(f"Unsupported architecture: {config.architecture}")


VTDynRandHetSNN = VTResNetSNN


def get_sigma_stats(model: nn.Module) -> dict[str, float] | None:
    sigmas = []
    for module in model.modules():
        if isinstance(module, AdaptiveRHLIFNode) and module.learnable_sigma:
            sigmas.append(module.sigma.item())
    if not sigmas:
        return None
    tensor = torch.tensor(sigmas, dtype=torch.float32)
    return {
        "mean": float(tensor.mean().item()),
        "std": float(tensor.std(unbiased=False).item()),
        "min": float(tensor.min().item()),
        "max": float(tensor.max().item()),
        "count": len(sigmas),
    }


## Metrics

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn

@dataclass
class ComputeMetrics:
    avg_synops: float
    avg_dense_ops: float
    avg_spike_count: float
    avg_spike_density: float
    activation_sparsity: float


class ComputeMonitor:
    def __init__(self, model: nn.Module) -> None:
        self.model = model
        self.handles: list[torch.utils.hooks.RemovableHandle] = []
        self.reset()

    def reset(self) -> None:
        self.synops = 0.0
        self.dense_ops = 0.0
        self.spike_count = 0.0
        self.spike_elements = 0.0
        self.zero_activations = 0.0
        self.total_activations = 0.0

    def _register(self) -> None:
        for module in self.model.modules():
            if isinstance(module, nn.Conv2d):
                self.handles.append(module.register_forward_hook(self._conv_hook))
            elif isinstance(module, nn.Linear):
                self.handles.append(module.register_forward_hook(self._linear_hook))
            elif getattr(module, "is_spiking_neuron", False):
                self.handles.append(module.register_forward_hook(self._spike_hook))

    def _remove(self) -> None:
        for handle in self.handles:
            handle.remove()
        self.handles.clear()

    def __enter__(self) -> "ComputeMonitor":
        self.reset()
        self._register()
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> None:
        self._remove()

    def _conv_hook(self, module: nn.Conv2d, inputs, output) -> None:
        x = inputs[0].detach()
        active_fraction = (x != 0).float().mean().item()
        output_elements = output.numel()
        kernel_mul = module.kernel_size[0] * module.kernel_size[1] * (module.in_channels // module.groups)
        dense_ops = output_elements * kernel_mul
        self.dense_ops += dense_ops
        self.synops += dense_ops * active_fraction
        self.zero_activations += (x == 0).sum().item()
        self.total_activations += x.numel()

    def _linear_hook(self, module: nn.Linear, inputs, output) -> None:
        x = inputs[0].detach()
        active_fraction = (x != 0).float().mean().item()
        dense_ops = x.size(0) * module.in_features * module.out_features
        self.dense_ops += dense_ops
        self.synops += dense_ops * active_fraction
        self.zero_activations += (x == 0).sum().item()
        self.total_activations += x.numel()

    def _spike_hook(self, module: nn.Module, inputs, output) -> None:
        spikes = output.detach()
        self.spike_count += spikes.sum().item()
        self.spike_elements += spikes.numel()

    def summary(self, batch_size: int) -> ComputeMetrics:
        denom = max(batch_size, 1)
        return ComputeMetrics(
            avg_synops=self.synops / denom,
            avg_dense_ops=self.dense_ops / denom,
            avg_spike_count=self.spike_count / denom,
            avg_spike_density=self.spike_count / max(self.spike_elements, 1.0),
            activation_sparsity=self.zero_activations / max(self.total_activations, 1.0),
        )


## Attacks

In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

try:
    import torchattacks
except Exception:
    # Kaggle and other hosted environments occasionally ship NumPy / SciPy wheels
    # that are ABI-incompatible with the current torchattacks dependency chain.
    # We only need torchattacks opportunistically; the suite has a fallback path
    # through the official autoattack package and our own PGD implementation.
    torchattacks = None

try:
    from autoattack import AutoAttack as ExternalAutoAttack
except ImportError:
    ExternalAutoAttack = None


def _untargeted_margin(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    true_logits = logits.gather(1, labels.unsqueeze(1)).squeeze(1)
    masked = logits.clone()
    masked.scatter_(1, labels.unsqueeze(1), float("-inf"))
    max_other = masked.max(dim=1).values
    return max_other - true_logits


def _build_attack_loss(
    trace: TraceResult,
    labels: torch.Tensor,
    stop_rule: StopRule | None,
    attack_config: AttackConfig,
) -> torch.Tensor:
    if attack_config.objective == "untargeted":
        return F.cross_entropy(trace.final_logits, labels)

    if attack_config.objective == "wrong_early_exit":
        if stop_rule is None:
            raise ValueError("wrong_early_exit objective requires an exit-aware stop rule.")
        losses = []
        total_steps = trace.avg_logits_per_step.size(0)
        for step in range(total_steps):
            logits = trace.avg_logits_per_step[step]
            margin = _untargeted_margin(logits, labels)
            confidence = stop_rule.confidence(logits)
            early_weight = (total_steps - step) / total_steps
            losses.append(early_weight * (margin + attack_config.wrong_exit_weight * confidence))
        return torch.stack(losses, dim=0).mean()

    if attack_config.objective == "compute_overuse":
        if stop_rule is None:
            raise ValueError("compute_overuse objective requires an exit-aware stop rule.")
        avg_logits = trace.avg_logits_per_step
        probabilities = F.softmax(avg_logits, dim=-1)
        entropy = -(probabilities * probabilities.clamp_min(1e-8).log()).sum(dim=-1)
        entropy = entropy / torch.log(torch.tensor(avg_logits.size(-1), device=avg_logits.device, dtype=avg_logits.dtype))
        confidences = torch.stack([stop_rule.confidence(step_logits) for step_logits in avg_logits], dim=0)
        delay_term = entropy.mean() - confidences.mean()
        preserve_term = -F.cross_entropy(trace.final_logits, labels)
        return attack_config.compute_weight * delay_term + attack_config.preserve_label_weight * preserve_term

    raise ValueError(f"Unsupported attack objective: {attack_config.objective}")


def _attack_progress(iterable, desc: str, total: int | None = None, leave: bool = False, enabled: bool = True):
    if tqdm is None or not enabled:
        return iterable
    return tqdm(iterable, total=total, desc=desc, leave=leave, dynamic_ncols=True)


def pgd_attack(
    model: TraceableSpikingModel | nn.Module,
    images: torch.Tensor,
    labels: torch.Tensor,
    attack_config: AttackConfig,
    time_steps: int,
    stop_rule: StopRule | None = None,
    threshold: float = 0.95,
) -> torch.Tensor:
    model.eval()
    base = images.detach()
    if attack_config.random_start:
        delta = torch.empty_like(base).uniform_(-attack_config.epsilon, attack_config.epsilon)
        delta = torch.clamp(base + delta, 0.0, 1.0) - base
    else:
        delta = torch.zeros_like(base)

    effective_time_steps = attack_config.attack_time_steps or time_steps

    show_step_progress = attack_config.steps > 1 and images.size(0) > 1
    step_iterator = _attack_progress(
        range(attack_config.steps),
        desc=f"PGD [{attack_config.objective}]",
        total=attack_config.steps,
        leave=False,
        enabled=show_step_progress,
    )
    for _ in step_iterator:
        eot_iters = max(1, attack_config.eot_iters)
        grad_accum = torch.zeros_like(delta)
        for _ in range(eot_iters):
            delta_step = delta.detach().requires_grad_(True)
            adv_images = torch.clamp(base + delta_step, 0.0, 1.0)
            trace = model.forward_trace(
                adv_images,
                time_steps=effective_time_steps,
                stop_rule=stop_rule if attack_config.exit_aware else None,
                threshold=threshold,
                force_full=attack_config.force_full_trace,
            )
            # Accumulate the EOT gradient one trace at a time. This is mathematically
            # equivalent to differentiating the mean EOT loss, but it avoids holding
            # all stochastic forward graphs in memory simultaneously.
            loss = _build_attack_loss(
                trace,
                labels,
                stop_rule if attack_config.exit_aware else None,
                attack_config,
            ) / float(eot_iters)
            grad = torch.autograd.grad(loss, delta_step, retain_graph=False, create_graph=False)[0]
            grad_accum = grad_accum + grad.detach()

            if hasattr(model, "reset"):
                model.reset()

            del delta_step, adv_images, trace, loss, grad

        delta = delta.detach() + attack_config.alpha * grad_accum.sign()
        delta = delta.clamp(-attack_config.epsilon, attack_config.epsilon)
        delta = torch.clamp(base + delta, 0.0, 1.0) - base

    return torch.clamp(base + delta.detach(), 0.0, 1.0)


class _ForwardTraceWrapper(nn.Module):
    def __init__(
        self,
        model: TraceableSpikingModel | nn.Module,
        time_steps: int,
        stop_rule: StopRule | None,
        threshold: float,
        exit_aware: bool,
        force_full_trace: bool,
        eot_iters: int,
    ) -> None:
        super().__init__()
        self.model = model
        self.time_steps = time_steps
        self.stop_rule = stop_rule if exit_aware else None
        self.threshold = threshold
        self.force_full_trace = force_full_trace
        self.eot_iters = max(1, eot_iters)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        logits = None
        for _ in range(self.eot_iters):
            trace = self.model.forward_trace(
                inputs,
                time_steps=self.time_steps,
                stop_rule=self.stop_rule,
                threshold=self.threshold,
                force_full=self.force_full_trace,
            )
            if logits is None:
                logits = trace.final_logits
            else:
                logits = logits + trace.final_logits
        assert logits is not None
        return logits / float(self.eot_iters)


def _resolve_autoattack_version(attack_config: AttackConfig) -> str:
    if attack_config.autoattack_version is not None:
        return attack_config.autoattack_version
    return "rand" if attack_config.eot_iters > 1 else "standard"


def autoattack_attack(
    model: TraceableSpikingModel | nn.Module,
    images: torch.Tensor,
    labels: torch.Tensor,
    attack_config: AttackConfig,
    time_steps: int,
    stop_rule: StopRule | None = None,
    threshold: float = 0.95,
) -> torch.Tensor:
    wrapper = _ForwardTraceWrapper(
        model=model,
        time_steps=attack_config.attack_time_steps or time_steps,
        stop_rule=stop_rule,
        threshold=threshold,
        exit_aware=attack_config.exit_aware,
        force_full_trace=attack_config.force_full_trace,
        eot_iters=attack_config.eot_iters,
    )
    wrapper.eval()
    version = _resolve_autoattack_version(attack_config)

    if torchattacks is not None:
        attack = torchattacks.AutoAttack(
            wrapper,
            norm="Linf",
            eps=attack_config.epsilon,
            version=version,
            n_classes=wrapper(images[:1]).size(1),
            seed=attack_config.autoattack_seed,
            verbose=attack_config.autoattack_verbose,
        )
        return attack(images, labels)

    if ExternalAutoAttack is not None:
        adversary = ExternalAutoAttack(
            wrapper,
            norm="Linf",
            eps=attack_config.epsilon,
            version=version,
            seed=attack_config.autoattack_seed,
            verbose=attack_config.autoattack_verbose,
        )
        try:
            return adversary.run_standard_evaluation(images, labels, bs=images.size(0), return_labels=False)
        except TypeError:
            return adversary.run_standard_evaluation(images, labels, bs=images.size(0))

    raise ImportError(
        "AutoAttack support requires either `torchattacks` or `autoattack` to be installed."
    )


## Training

In [ ]:
from __future__ import annotations

import gc
import math
import random
from contextlib import nullcontext
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


try:
    from torch.amp import GradScaler
except ImportError:
    from torch.cuda.amp import GradScaler

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _autocast_context(config: TrainingConfig):
    device_type = "cuda" if str(config.device).startswith("cuda") else "cpu"
    enabled = config.use_amp and device_type == "cuda"
    if not enabled:
        return nullcontext()
    return torch.autocast(device_type=device_type, dtype=getattr(torch, config.amp_dtype))


def _build_grad_scaler(enabled: bool):
    try:
        return GradScaler("cuda", enabled=enabled)
    except TypeError:
        return GradScaler(enabled=enabled)


def _clean_ratio(epoch: int, total_epochs: int, config: TrainingConfig) -> float:
    if total_epochs <= 1:
        return config.clean_ratio_end
    progress = (epoch - 1) / (total_epochs - 1)
    return config.clean_ratio_start + (config.clean_ratio_end - config.clean_ratio_start) * progress


def _sample_time_steps(config: TrainingConfig) -> int:
    return random.choice(config.mixed_time_steps)


def _training_maybe_len(iterable) -> int | None:
    try:
        return len(iterable)
    except TypeError:
        return None


def _training_progress(iterable, desc: str, total: int | None = None, leave: bool = False):
    if tqdm is None:
        return iterable
    resolved_total = _training_maybe_len(iterable) if total is None else total
    return tqdm(iterable, total=resolved_total, desc=desc, leave=leave, dynamic_ncols=True)


def _release_memory(
    device: str,
    model: TraceableSpikingModel | None = None,
    optimizer: torch.optim.Optimizer | None = None,
) -> None:
    if model is not None:
        model.reset()
    if optimizer is not None:
        optimizer.zero_grad(set_to_none=True)
    gc.collect()
    if str(device).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.empty_cache()


def _entropy_regularization(model: nn.Module, target_sigma: float = 0.5) -> torch.Tensor:
    total = None
    count = 0
    for module in model.modules():
        if isinstance(module, AdaptiveRHLIFNode) and module.learnable_sigma:
            term = torch.clamp(target_sigma - module.sigma, min=0.0) ** 2
            total = term if total is None else total + term
            count += 1
    if total is None:
        return torch.tensor(0.0, device=next(model.parameters()).device)
    return total / max(count, 1)


def parseval_retraction(model: nn.Module, beta: float = 0.004) -> None:
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                weight = module.weight.data
                shape = weight.shape
                flat = weight.view(shape[0], -1)
                update = (1 + beta) * flat - beta * flat @ flat.t() @ flat
                module.weight.data.copy_(update.view(shape))


class AdversarialWeightPerturbation:
    def __init__(self, model: nn.Module, adv_param: str = "weight", adv_lr: float = 0.01, adv_eps: float = 0.01):
        self.model = model
        self.adv_param = adv_param
        self.adv_lr = adv_lr
        self.adv_eps = adv_eps
        self.backup: dict[str, torch.Tensor] = {}

    def perturb(self) -> None:
        eps = 1e-6
        for name, param in self.model.named_parameters():
            if self.adv_param not in name or param.grad is None or not param.requires_grad:
                continue
            if name not in self.backup:
                self.backup[name] = param.data.clone()
            grad_norm = torch.norm(param.grad)
            weight_norm = torch.norm(param.data.detach())
            if grad_norm == 0 or torch.isnan(grad_norm):
                continue
            step = self.adv_lr * param.grad / (grad_norm + eps) * (weight_norm + eps)
            step = torch.clamp(step, -self.adv_eps, self.adv_eps)
            param.data.add_(step)

    def restore(self) -> None:
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup.clear()


def build_optimizer(model: TraceableSpikingModel, config: TrainingConfig) -> torch.optim.Optimizer:
    sigma_params = []
    other_params = []
    for name, param in model.named_parameters():
        if "sigma" in name:
            sigma_params.append(param)
        else:
            other_params.append(param)
    return torch.optim.SGD(
        [
            {"params": other_params, "lr": config.base_lr},
            {"params": sigma_params, "lr": config.base_lr * config.sigma_lr_scale},
        ],
        momentum=config.momentum,
        weight_decay=config.weight_decay,
    )


def build_scheduler(optimizer: torch.optim.Optimizer, config: TrainingConfig):
    def lr_lambda(epoch_idx: int) -> float:
        if epoch_idx < config.warmup_epochs:
            return float(epoch_idx + 1) / max(config.warmup_epochs, 1)
        progress = (epoch_idx - config.warmup_epochs) / max(config.epochs - config.warmup_epochs, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def _build_rfgsm_batch(
    model: TraceableSpikingModel,
    data: torch.Tensor,
    labels: torch.Tensor,
    time_steps: int,
    config: TrainingConfig,
) -> torch.Tensor:
    delta = torch.zeros_like(data).uniform_(
        -config.epsilon * config.random_start_scale,
        config.epsilon * config.random_start_scale,
    )
    delta.requires_grad_(True)
    adv_inputs = torch.clamp(data + delta, 0.0, 1.0)
    with _autocast_context(config):
        logits = model(adv_inputs, time_steps=time_steps, use_vt=False)
        loss = F.cross_entropy(logits, labels)
    grad = torch.autograd.grad(loss, delta, only_inputs=True)[0]
    delta = torch.clamp(delta.detach() + config.alpha * grad.sign(), -config.epsilon, config.epsilon)
    return torch.clamp(data + delta, 0.0, 1.0)


def train_one_epoch(
    model: TraceableSpikingModel,
    train_loader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    config: TrainingConfig,
    epoch: int,
    awp: AdversarialWeightPerturbation | None = None,
) -> dict[str, float]:
    model.train()
    criterion = nn.CrossEntropyLoss()
    stats = {"loss": 0.0, "clean_correct": 0.0, "adv_correct": 0.0, "clean_total": 0.0, "adv_total": 0.0}
    clean_ratio = _clean_ratio(epoch, config.epochs, config)

    batch_iterator = _training_progress(train_loader, desc=f"Epoch {epoch}/{config.epochs}", leave=False)
    for batch_idx, (data, labels) in enumerate(batch_iterator):
        data = data.to(config.device, non_blocking=True)
        labels = labels.to(config.device, non_blocking=True)
        batch_size = data.size(0)
        time_steps = _sample_time_steps(config)

        n_clean = int(batch_size * clean_ratio)
        perm = torch.randperm(batch_size, device=config.device)
        clean_idx = perm[:n_clean]
        adv_idx = perm[n_clean:]

        clean_data = data[clean_idx]
        clean_labels = labels[clean_idx]
        adv_data = data[adv_idx]
        adv_labels = labels[adv_idx]

        adv_batch = None
        if adv_idx.numel() > 0:
            adv_batch = _build_rfgsm_batch(model, adv_data, adv_labels, time_steps, config)

        optimizer.zero_grad(set_to_none=True)
        with _autocast_context(config):
            clean_loss = torch.tensor(0.0, device=config.device)
            adv_loss = torch.tensor(0.0, device=config.device)
            clean_logits = None
            adv_logits = None

            if clean_idx.numel() > 0:
                clean_logits = model(clean_data, time_steps=time_steps, use_vt=False)
                clean_loss = criterion(clean_logits, clean_labels)
            if adv_idx.numel() > 0 and adv_batch is not None:
                adv_logits = model(adv_batch, time_steps=time_steps, use_vt=False)
                adv_loss = criterion(adv_logits, adv_labels)
            main_loss = clean_ratio * clean_loss + (1.0 - clean_ratio) * adv_loss
            total_loss = main_loss + config.entropy_lambda * _entropy_regularization(model)

        scaler.scale(total_loss).backward()

        if awp is not None:
            awp.perturb()
            optimizer.zero_grad(set_to_none=True)
            with _autocast_context(config):
                awp_clean_loss = torch.tensor(0.0, device=config.device)
                awp_adv_loss = torch.tensor(0.0, device=config.device)
                if clean_idx.numel() > 0:
                    awp_clean_loss = criterion(model(clean_data, time_steps=time_steps, use_vt=False), clean_labels)
                if adv_idx.numel() > 0 and adv_batch is not None:
                    awp_adv_loss = criterion(model(adv_batch, time_steps=time_steps, use_vt=False), adv_labels)
                awp_loss = clean_ratio * awp_clean_loss + (1.0 - clean_ratio) * awp_adv_loss
            scaler.scale(awp_loss).backward()
            awp.restore()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.gradient_clip)
        scaler.step(optimizer)
        scaler.update()

        if config.use_parseval and batch_idx % config.parseval_interval == 0:
            parseval_retraction(model, beta=config.parseval_beta)

        stats["loss"] += total_loss.item() * batch_size
        if clean_logits is not None:
            stats["clean_correct"] += clean_logits.argmax(dim=1).eq(clean_labels).sum().item()
            stats["clean_total"] += clean_labels.size(0)
        if adv_logits is not None:
            stats["adv_correct"] += adv_logits.argmax(dim=1).eq(adv_labels).sum().item()
            stats["adv_total"] += adv_labels.size(0)

        if hasattr(batch_iterator, "set_postfix"):
            batch_iterator.set_postfix(
                loss=f"{stats['loss'] / max(stats['clean_total'] + stats['adv_total'], 1.0):.4f}",
                clean=f"{100.0 * stats['clean_correct'] / max(stats['clean_total'], 1.0):.2f}",
                adv=f"{100.0 * stats['adv_correct'] / max(stats['adv_total'], 1.0):.2f}",
                T=time_steps,
            )

    total_samples = max(stats["clean_total"] + stats["adv_total"], 1.0)
    return {
        "loss": stats["loss"] / total_samples,
        "train_clean_acc": 100.0 * stats["clean_correct"] / max(stats["clean_total"], 1.0),
        "train_adv_acc": 100.0 * stats["adv_correct"] / max(stats["adv_total"], 1.0),
        "clean_ratio": clean_ratio,
    }


def _evaluate_selection_metrics(
    model: TraceableSpikingModel,
    val_loader,
    config: TrainingConfig,
    selection_attack_config: AttackConfig | None = None,
) -> tuple[float, float]:

    max_time_steps = max(config.mixed_time_steps)
    clean_metrics = evaluate_policy(
        model,
        val_loader,
        mode="fixed",
        time_steps=max_time_steps,
        compute_metrics=False,
        exact_vt_compute=False,
        device=config.device,
        progress_desc="Validation clean",
    )
    robust_accuracy = 0.0
    if selection_attack_config is not None and config.selection_metric in {"robust", "clean_plus_robust"}:
        robust_metrics = evaluate_policy(
            model,
            val_loader,
            mode="fixed",
            time_steps=max_time_steps,
            attack_config=selection_attack_config,
            compute_metrics=False,
            exact_vt_compute=False,
            device=config.device,
            progress_desc="Validation robust",
        )
        robust_accuracy = robust_metrics["accuracy"]
    return clean_metrics["accuracy"], robust_accuracy


def _selection_score(clean_accuracy: float, robust_accuracy: float, metric: str) -> float:
    if metric == "clean":
        return clean_accuracy
    if metric == "robust":
        return robust_accuracy
    if metric == "clean_plus_robust":
        return clean_accuracy + robust_accuracy
    raise ValueError(f"Unsupported selection_metric: {metric}")


def fit_model(
    model: TraceableSpikingModel,
    train_loader,
    val_loader,
    config: TrainingConfig,
    use_awp: bool = False,
    selection_attack_config: AttackConfig | None = None,
) -> tuple[dict[str, list[float]], Path]:
    model.to(config.device)
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config)
    scaler = _build_grad_scaler(config.use_amp and str(config.device).startswith("cuda"))
    awp = AdversarialWeightPerturbation(model) if use_awp else None

    output_dir = ensure_output_dir(config)
    checkpoint_path = output_dir / config.checkpoint_name
    history = {
        "loss": [],
        "train_clean_acc": [],
        "train_adv_acc": [],
        "val_clean_acc": [],
        "val_robust_acc": [],
        "sigma_mean": [],
        "lr": [],
    }

    best_state = None
    best_score = float("-inf")
    best_clean = 0.0
    best_robust = 0.0

    epoch_iterator = _training_progress(range(1, config.epochs + 1), desc="Training epochs", leave=True)
    for epoch in epoch_iterator:
        train_stats = train_one_epoch(model, train_loader, optimizer, scaler, config, epoch, awp=awp)
        scheduler.step()
        validation_ran = epoch % config.eval_interval == 0
        val_clean, val_robust = (float("nan"), float("nan"))
        if validation_ran:
            _release_memory(config.device, model=model, optimizer=optimizer)
            val_clean, val_robust = _evaluate_selection_metrics(
                model,
                val_loader,
                config,
                selection_attack_config=selection_attack_config,
            )
            _release_memory(config.device, model=model, optimizer=optimizer)

        sigma_stats = get_sigma_stats(model) or {"mean": 0.0}

        history["loss"].append(train_stats["loss"])
        history["train_clean_acc"].append(train_stats["train_clean_acc"])
        history["train_adv_acc"].append(train_stats["train_adv_acc"])
        history["val_clean_acc"].append(val_clean)
        history["val_robust_acc"].append(val_robust)
        history["sigma_mean"].append(sigma_stats["mean"])
        history["lr"].append(optimizer.param_groups[0]["lr"])

        if validation_ran:
            score = _selection_score(val_clean, val_robust, config.selection_metric)
        else:
            score = None

        if validation_ran and score is not None and score >= best_score:
            best_score = score
            best_clean = val_clean
            best_robust = val_robust
            best_state = {key: value.detach().cpu() for key, value in model.state_dict().items()}

        if hasattr(epoch_iterator, "set_postfix"):
            epoch_iterator.set_postfix(
                loss=f"{train_stats['loss']:.4f}",
                clean=f"{train_stats['train_clean_acc']:.2f}",
                adv=f"{train_stats['train_adv_acc']:.2f}",
                val=f"{val_clean:.2f}" if validation_ran else "skip",
                robust=f"{val_robust:.2f}" if validation_ran else "skip",
                best=f"{best_score:.2f}" if best_score != float("-inf") else "n/a",
            )

    if best_state is None:
        best_state = {key: value.detach().cpu() for key, value in model.state_dict().items()}

    torch.save(
        {
            "model_state_dict": best_state,
            "history": history,
            "training_config": asdict(config),
            "best_val_clean_acc": best_clean,
            "best_val_robust_acc": best_robust,
            "selection_metric": config.selection_metric,
        },
        checkpoint_path,
    )
    model.load_state_dict(best_state)
    return history, checkpoint_path


## Experiments

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from math import sqrt
from typing import Iterable

import pandas as pd
import torch


try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None


def _experiments_maybe_len(iterable) -> int | None:
    try:
        return len(iterable)
    except TypeError:
        return None


def _experiments_progress(iterable, desc: str, total: int | None = None, leave: bool = False):
    if tqdm is None:
        return iterable
    resolved_total = _experiments_maybe_len(iterable) if total is None else total
    return tqdm(iterable, total=resolved_total, desc=desc, leave=leave, dynamic_ncols=True)


def _iter_batches(dataloader, max_batches: int | None, desc: str, leave: bool = False):
    total = _experiments_maybe_len(dataloader)
    if total is not None and max_batches is not None:
        total = min(total, max_batches)
    iterator = _experiments_progress(dataloader, desc=desc, total=total, leave=leave)
    for batch_idx, batch in enumerate(iterator):
        if max_batches is not None and batch_idx >= max_batches:
            break
        yield batch


def _run_trace(
    model: TraceableSpikingModel,
    inputs: torch.Tensor,
    mode: str,
    time_steps: int,
    stop_rule: StopRule | None,
    threshold: float,
    force_full: bool = False,
) -> TraceResult:
    if mode == "fixed":
        return model.forward_trace(inputs, time_steps=time_steps, force_full=force_full)
    return model.forward_trace(
        inputs,
        time_steps=time_steps,
        stop_rule=stop_rule,
        threshold=threshold,
        force_full=force_full,
    )


def _empty_result(mode: str, time_steps: int, stop_rule: StopRule | None, threshold: float, attack_config: AttackConfig | None, compute_mode: str) -> dict[str, object]:
    autoattack_version = None
    if attack_config is not None and attack_config.attack_name == "autoattack":
        autoattack_version = attack_config.autoattack_version or ("rand" if attack_config.eot_iters > 1 else "standard")
    return {
        "mode": mode,
        "time_steps": time_steps,
        "stop_rule": stop_rule.name if stop_rule is not None else "fixed",
        "threshold": threshold if mode == "vt" else None,
        "accuracy": 0.0,
        "avg_steps": 0.0,
        "avg_stopping_depth": 0.0,
        "avg_executed_steps": 0.0,
        "avg_confidence": 0.0,
        "step_histogram": {},
        "compute_mode": compute_mode,
        "attack_name": attack_config.attack_name if attack_config is not None else "clean",
        "attack_objective": attack_config.objective if attack_config is not None else "clean",
        "attack_steps": attack_config.steps if attack_config is not None else 0,
        "attack_eot": attack_config.eot_iters if attack_config is not None else 0,
        "exit_aware_attack": bool(attack_config.exit_aware) if attack_config is not None else False,
        "autoattack_version": autoattack_version,
    }


def _make_attack_inputs(
    model: TraceableSpikingModel,
    data: torch.Tensor,
    labels: torch.Tensor,
    attack_config: AttackConfig | None,
    time_steps: int,
    stop_rule: StopRule | None,
    threshold: float,
) -> torch.Tensor:
    if attack_config is None:
        return data
    if attack_config.attack_name == "pgd":
        return pgd_attack(model, data, labels, attack_config, time_steps, stop_rule=stop_rule, threshold=threshold)
    if attack_config.attack_name == "autoattack":
        return autoattack_attack(model, data, labels, attack_config, time_steps, stop_rule=stop_rule, threshold=threshold)
    raise ValueError(f"Unsupported attack_name: {attack_config.attack_name}")


def _update_metrics(totals: defaultdict[float, float], summary, weight: int) -> None:
    totals["synops"] += summary.avg_synops * weight
    totals["dense_ops"] += summary.avg_dense_ops * weight
    totals["spike_count"] += summary.avg_spike_count * weight
    totals["spike_density"] += summary.avg_spike_density * weight
    totals["activation_sparsity"] += summary.activation_sparsity * weight


def evaluate_policy(
    model: TraceableSpikingModel,
    dataloader,
    mode: str,
    time_steps: int,
    stop_rule: StopRule | None = None,
    threshold: float = 0.95,
    attack_config: AttackConfig | None = None,
    compute_metrics: bool = False,
    exact_vt_compute: bool = True,
    device: str = "cuda",
    max_batches: int | None = None,
    force_full_trace: bool = False,
    progress_desc: str | None = None,
) -> dict[str, object]:
    model.eval()
    totals: defaultdict[str, float] = defaultdict(float)
    step_histogram: Counter[int] = Counter()
    confidence_rule = stop_rule or build_stop_rule("max_softmax")
    compute_mode = "sample_exact" if compute_metrics and mode == "vt" and exact_vt_compute else "batched_proxy"
    attack_name = attack_config.attack_name if attack_config is not None else "clean"
    desc = progress_desc or f"Eval {mode} [{attack_name}]"

    for data, labels in _iter_batches(dataloader, max_batches, desc=desc, leave=False):
        data = data.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if compute_mode == "sample_exact":
            sample_iterator = _experiments_progress(range(data.size(0)), desc=f"{desc} samples", total=data.size(0), leave=False)
            for sample_idx in sample_iterator:
                single_data = data[sample_idx:sample_idx + 1]
                single_label = labels[sample_idx:sample_idx + 1]
                inputs = (
                    _make_attack_inputs(model, single_data, single_label, attack_config, time_steps, stop_rule, threshold)
                    if attack_config is not None
                    else single_data
                )
                with ComputeMonitor(model) as monitor:
                    with torch.no_grad():
                        trace = _run_trace(
                            model,
                            inputs,
                            mode=mode,
                            time_steps=time_steps,
                            stop_rule=stop_rule,
                            threshold=threshold,
                            force_full=force_full_trace,
                        )
                summary = monitor.summary(1)
                _update_metrics(totals, summary, 1)
                prediction = trace.final_logits.argmax(dim=1)
                totals["correct"] += prediction.eq(single_label).sum().item()
                totals["count"] += 1
                totals["stopping_depth"] += trace.steps_taken.item()
                totals["executed_steps"] += trace.executed_steps
                totals["confidence"] += confidence_rule.confidence(trace.final_logits).sum().item()
                step_histogram.update(trace.steps_taken.detach().cpu().tolist())
            continue

        inputs = _make_attack_inputs(model, data, labels, attack_config, time_steps, stop_rule, threshold)

        if compute_metrics:
            with ComputeMonitor(model) as monitor:
                with torch.no_grad():
                    trace = _run_trace(
                        model,
                        inputs,
                        mode=mode,
                        time_steps=time_steps,
                        stop_rule=stop_rule,
                        threshold=threshold,
                        force_full=force_full_trace,
                    )
            summary = monitor.summary(labels.size(0))
            _update_metrics(totals, summary, labels.size(0))
        else:
            with torch.no_grad():
                trace = _run_trace(
                    model,
                    inputs,
                    mode=mode,
                    time_steps=time_steps,
                    stop_rule=stop_rule,
                    threshold=threshold,
                    force_full=force_full_trace,
                )

        predictions = trace.final_logits.argmax(dim=1)
        totals["correct"] += predictions.eq(labels).sum().item()
        totals["count"] += labels.size(0)
        totals["stopping_depth"] += trace.steps_taken.sum().item()
        totals["executed_steps"] += trace.executed_steps * labels.size(0)
        totals["confidence"] += confidence_rule.confidence(trace.final_logits).sum().item()
        step_histogram.update(trace.steps_taken.detach().cpu().tolist())

    count = max(totals["count"], 1.0)
    result = _empty_result(mode, time_steps, stop_rule, threshold, attack_config, compute_mode)
    result.update(
        {
            "accuracy": 100.0 * totals["correct"] / count,
            "avg_steps": totals["stopping_depth"] / count,
            "avg_stopping_depth": totals["stopping_depth"] / count,
            "avg_executed_steps": totals["executed_steps"] / count,
            "avg_confidence": totals["confidence"] / count,
            "step_histogram": dict(sorted(step_histogram.items())),
        }
    )
    if compute_metrics:
        result.update(
            {
                "avg_synops": totals["synops"] / count,
                "avg_dense_ops": totals["dense_ops"] / count,
                "avg_spike_count": totals["spike_count"] / count,
                "avg_spike_density": totals["spike_density"] / count,
                "activation_sparsity": totals["activation_sparsity"] / count,
            }
        )
    model.reset()
    return result


def threshold_sweep(
    model: TraceableSpikingModel,
    dataloader,
    thresholds: Iterable[float],
    stop_rule: StopRule,
    time_steps: int,
    attack_config: AttackConfig | None = None,
    compute_metrics: bool = False,
    exact_vt_compute: bool = True,
    device: str = "cuda",
    max_batches: int | None = None,
) -> pd.DataFrame:
    threshold_values = list(thresholds)
    rows = []
    for threshold in _experiments_progress(threshold_values, desc=f"VT threshold sweep [{stop_rule.name}]", total=len(threshold_values), leave=False):
        rows.append(
            evaluate_policy(
            model=model,
            dataloader=dataloader,
            mode="vt",
            time_steps=time_steps,
            stop_rule=stop_rule,
            threshold=threshold,
            attack_config=attack_config,
            compute_metrics=compute_metrics,
            exact_vt_compute=exact_vt_compute,
            device=device,
            max_batches=max_batches,
            progress_desc=f"VT eval [{stop_rule.name}] theta={threshold:.2f}",
        )
        )
    return pd.DataFrame(rows)


def fixed_time_sweep(
    model: TraceableSpikingModel,
    dataloader,
    time_step_values: Iterable[int],
    attack_config: AttackConfig | None = None,
    compute_metrics: bool = False,
    device: str = "cuda",
    max_batches: int | None = None,
) -> pd.DataFrame:
    step_values = list(time_step_values)
    rows = []
    for time_steps in _experiments_progress(step_values, desc="Fixed-T sweep", total=len(step_values), leave=False):
        rows.append(
            evaluate_policy(
            model=model,
            dataloader=dataloader,
            mode="fixed",
            time_steps=time_steps,
            attack_config=attack_config,
            compute_metrics=compute_metrics,
            exact_vt_compute=False,
            device=device,
            max_batches=max_batches,
            progress_desc=f"Fixed-T eval T={time_steps}",
        )
        )
    return pd.DataFrame(rows)


def _pick_match_column(vt_results: pd.DataFrame, fixed_results: pd.DataFrame, match_metric: str | None) -> str:
    if match_metric is not None:
        return match_metric
    if "avg_synops" in vt_results.columns and "avg_synops" in fixed_results.columns:
        return "avg_synops"
    if "avg_executed_steps" in vt_results.columns and "avg_executed_steps" in fixed_results.columns:
        return "avg_executed_steps"
    return "avg_stopping_depth"


def match_fixed_time_controls(
    vt_results: pd.DataFrame,
    fixed_results: pd.DataFrame,
    match_metric: str | None = None,
) -> pd.DataFrame:
    matches = []
    metric = _pick_match_column(vt_results, fixed_results, match_metric)
    for _, row in vt_results.iterrows():
        distances = (fixed_results[metric] - row[metric]).abs()
        fixed_row = fixed_results.loc[distances.idxmin()]
        match = {
            "match_metric": metric,
            "stop_rule": row["stop_rule"],
            "threshold": row["threshold"],
            "vt_accuracy": row["accuracy"],
            "vt_avg_stopping_depth": row["avg_stopping_depth"],
            "vt_avg_executed_steps": row.get("avg_executed_steps"),
            "matched_fixed_t": fixed_row["time_steps"],
            "fixed_accuracy": fixed_row["accuracy"],
            "fixed_avg_executed_steps": fixed_row.get("avg_executed_steps"),
            "metric_gap": abs(fixed_row[metric] - row[metric]),
            "accuracy_delta": row["accuracy"] - fixed_row["accuracy"],
        }
        if "avg_synops" in row and "avg_synops" in fixed_row:
            match["vt_avg_synops"] = row["avg_synops"]
            match["fixed_avg_synops"] = fixed_row["avg_synops"]
            match["synops_delta"] = row["avg_synops"] - fixed_row["avg_synops"]
        matches.append(match)
    return pd.DataFrame(matches)


def select_threshold_neighbors(
    vt_results: pd.DataFrame,
    selected_threshold: float,
    count: int = 3,
) -> pd.DataFrame:
    if "threshold" not in vt_results.columns:
        raise ValueError("VT results must include a `threshold` column.")
    unique_thresholds = sorted({float(value) for value in vt_results["threshold"].dropna().tolist()})
    if not unique_thresholds:
        return pd.DataFrame(columns=["threshold", "threshold_role", "distance_from_selected", "selection_rank"])

    target_count = max(1, min(count, len(unique_thresholds)))
    anchor_idx = min(
        range(len(unique_thresholds)),
        key=lambda idx: abs(unique_thresholds[idx] - float(selected_threshold)),
    )

    chosen_indices = [anchor_idx]
    step = 1
    while len(chosen_indices) < target_count:
        left_idx = anchor_idx - step
        right_idx = anchor_idx + step
        if left_idx >= 0:
            chosen_indices.append(left_idx)
            if len(chosen_indices) >= target_count:
                break
        if right_idx < len(unique_thresholds):
            chosen_indices.append(right_idx)
            if len(chosen_indices) >= target_count:
                break
        step += 1

    rows = []
    for rank, idx in enumerate(sorted(chosen_indices, key=lambda value: unique_thresholds[value])):
        threshold = unique_thresholds[idx]
        if abs(threshold - float(selected_threshold)) < 1e-12:
            role = "selected"
        elif threshold < float(selected_threshold):
            role = "lower_neighbor"
        else:
            role = "upper_neighbor"
        rows.append(
            {
                "threshold": threshold,
                "threshold_role": role,
                "distance_from_selected": abs(threshold - float(selected_threshold)),
                "selection_rank": rank,
            }
        )
    return pd.DataFrame(rows)


def select_compute_matched_threshold(
    model: TraceableSpikingModel,
    dataloader,
    thresholds: Iterable[float],
    fixed_time_steps: Iterable[int],
    stop_rule: StopRule,
    time_steps: int,
    attack_config: AttackConfig | None = None,
    compute_metrics: bool = False,
    exact_vt_compute: bool = True,
    match_metric: str | None = None,
    device: str = "cuda",
    max_batches: int | None = None,
    vt_dataloader=None,
    fixed_dataloader=None,
) -> dict[str, object]:
    vt_loader = dataloader if vt_dataloader is None else vt_dataloader
    fixed_loader = dataloader if fixed_dataloader is None else fixed_dataloader
    if vt_loader is None or fixed_loader is None:
        raise ValueError("select_compute_matched_threshold requires at least one dataloader.")
    vt_results = threshold_sweep(
        model,
        vt_loader,
        thresholds=thresholds,
        stop_rule=stop_rule,
        time_steps=time_steps,
        attack_config=attack_config,
        compute_metrics=compute_metrics,
        exact_vt_compute=exact_vt_compute,
        device=device,
        max_batches=max_batches,
    )
    fixed_results = fixed_time_sweep(
        model,
        fixed_loader,
        time_step_values=fixed_time_steps,
        attack_config=attack_config,
        compute_metrics=compute_metrics,
        device=device,
        max_batches=max_batches,
    )
    matches = match_fixed_time_controls(vt_results, fixed_results, match_metric=match_metric)
    if matches.empty:
        raise RuntimeError("No VT/fixed-T matches were produced during threshold calibration.")
    sort_cols = [column for column in ["accuracy_delta", "synops_delta", "metric_gap"] if column in matches.columns]
    if not sort_cols:
        raise RuntimeError("Threshold calibration produced matches without sortable comparison columns.")
    ascending = [False if column == "accuracy_delta" else True for column in sort_cols]
    selected_match = matches.sort_values(sort_cols, ascending=ascending).iloc[0]
    effective_match_metric = selected_match["match_metric"]
    selected_threshold = float(selected_match["threshold"])
    metric_value = float(vt_results.loc[vt_results["threshold"] == selected_threshold, effective_match_metric].iloc[0])
    return {
        "selected_threshold": selected_threshold,
        "selected_metric_value": metric_value,
        "selected_match": selected_match.to_dict(),
        "vt_results": vt_results,
        "fixed_results": fixed_results,
        "matches": matches,
    }


def calibrate_threshold(
    model: TraceableSpikingModel,
    dataloader,
    stop_rule: StopRule,
    candidate_thresholds: Iterable[float],
    target_value: float,
    time_steps: int,
    attack_config: AttackConfig | None = None,
    compute_metrics: bool = False,
    exact_vt_compute: bool = True,
    target_metric: str = "avg_stopping_depth",
    device: str = "cuda",
    max_batches: int | None = None,
) -> dict[str, object]:
    best_result = None
    best_gap = None
    threshold_values = list(candidate_thresholds)
    for threshold in _experiments_progress(threshold_values, desc=f"Calibrate [{stop_rule.name}]", total=len(threshold_values), leave=False):
        result = evaluate_policy(
            model=model,
            dataloader=dataloader,
            mode="vt",
            time_steps=time_steps,
            stop_rule=stop_rule,
            threshold=threshold,
            attack_config=attack_config,
            compute_metrics=compute_metrics,
            exact_vt_compute=exact_vt_compute,
            device=device,
            max_batches=max_batches,
            progress_desc=f"Calib eval [{stop_rule.name}] theta={threshold:.2f}",
        )
        gap = abs(result[target_metric] - target_value)
        if best_result is None or gap < best_gap:
            best_result = result
            best_gap = gap
    assert best_result is not None
    best_result["target_metric"] = target_metric
    best_result["target_value"] = target_value
    best_result["metric_gap"] = best_gap
    return best_result


def compare_stopping_rules(
    model: TraceableSpikingModel,
    dataloader,
    rule_names: Iterable[str],
    candidate_thresholds: Iterable[float],
    time_steps: int,
    target_value: float | None = None,
    attack_config: AttackConfig | None = None,
    compute_metrics: bool = False,
    exact_vt_compute: bool = True,
    target_metric: str = "avg_stopping_depth",
    device: str = "cuda",
    max_batches: int | None = None,
) -> pd.DataFrame:
    rows = []
    rule_values = list(rule_names)
    for rule_name in _experiments_progress(rule_values, desc="Stopping-rule comparison", total=len(rule_values), leave=False):
        stop_rule = build_stop_rule(rule_name)
        if target_value is None:
            sweep = threshold_sweep(
                model,
                dataloader,
                candidate_thresholds,
                stop_rule,
                time_steps,
                attack_config=attack_config,
                compute_metrics=compute_metrics,
                exact_vt_compute=exact_vt_compute,
                device=device,
                max_batches=max_batches,
            )
            rows.extend(sweep.to_dict("records"))
        else:
            rows.append(
                calibrate_threshold(
                    model,
                    dataloader,
                    stop_rule,
                    candidate_thresholds,
                    target_value,
                    time_steps,
                    attack_config=attack_config,
                    compute_metrics=compute_metrics,
                    exact_vt_compute=exact_vt_compute,
                    target_metric=target_metric,
                    device=device,
                    max_batches=max_batches,
                )
            )
    return pd.DataFrame(rows)


def collect_temporal_diagnostics(
    model: TraceableSpikingModel,
    dataloader,
    time_steps: int,
    stop_rule: StopRule | None = None,
    threshold: float = 0.95,
    attack_config: AttackConfig | None = None,
    device: str = "cuda",
    max_batches: int | None = None,
) -> dict[str, object]:
    model.eval()
    confidence_rule = stop_rule or build_stop_rule("max_softmax")
    confidence_sums = None
    accuracy_sums = None
    counts = None
    step_histogram: Counter[int] = Counter()

    for data, labels in _iter_batches(dataloader, max_batches, desc="Temporal diagnostics", leave=False):
        data = data.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        inputs = _make_attack_inputs(model, data, labels, attack_config, time_steps, stop_rule, threshold)
        with torch.no_grad():
            trace = _run_trace(
                model,
                inputs,
                mode="vt" if stop_rule is not None else "fixed",
                time_steps=time_steps,
                stop_rule=stop_rule,
                threshold=threshold,
                force_full=True,
            )

        steps = trace.avg_logits_per_step.size(0)
        if confidence_sums is None:
            confidence_sums = torch.zeros(steps, dtype=torch.float64)
            accuracy_sums = torch.zeros(steps, dtype=torch.float64)
            counts = torch.zeros(steps, dtype=torch.float64)

        for step in range(steps):
            step_logits = trace.avg_logits_per_step[step]
            confidence_sums[step] += confidence_rule.confidence(step_logits).sum().item()
            accuracy_sums[step] += step_logits.argmax(dim=1).eq(labels).sum().item()
            counts[step] += labels.size(0)
        step_histogram.update(trace.steps_taken.detach().cpu().tolist())

    model.reset()
    if confidence_sums is None or accuracy_sums is None or counts is None:
        return {"avg_confidence_by_step": [], "avg_accuracy_by_step": [], "step_histogram": {}}

    return {
        "avg_confidence_by_step": (confidence_sums / counts.clamp(min=1)).tolist(),
        "avg_accuracy_by_step": (100.0 * accuracy_sums / counts.clamp(min=1)).tolist(),
        "step_histogram": dict(sorted(step_histogram.items())),
    }


def summarize_multiseed(records: list[dict[str, object]], group_cols: list[str]) -> pd.DataFrame:
    if not records:
        return pd.DataFrame()
    frame = pd.DataFrame(records)
    numeric_cols = [column for column in frame.columns if pd.api.types.is_numeric_dtype(frame[column])]
    summaries = []
    for keys, group in frame.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {column: value for column, value in zip(group_cols, keys)}
        for column in numeric_cols:
            values = group[column].dropna().astype(float)
            if values.empty:
                continue
            mean = values.mean()
            std = values.std(ddof=1) if len(values) > 1 else 0.0
            ci95 = 1.96 * std / sqrt(len(values)) if len(values) > 1 else 0.0
            row[f"{column}_mean"] = mean
            row[f"{column}_std"] = std
            row[f"{column}_ci95"] = ci95
        summaries.append(row)
    return pd.DataFrame(summaries)


# Experiment Driver

This driver reuses the full VT training and evaluation workflow, but the defaults are specialized for a deterministic vanilla ResNet-18 SNN.

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:
from dataclasses import replace

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

DEFAULT_OUTPUT_ROOT = Path("/kaggle/working/vt_vanilla_resnet18_outputs") if Path("/kaggle/working").exists() else Path("vt_vanilla_resnet18_outputs")
DEFAULT_DATA_ROOT = os.environ.get(
    "VT_OFFLINE_CIFAR_ROOT",
    "/kaggle/working/data" if Path("/kaggle/working").exists() else "./data",
)
EXPERIMENT_ROOT = DEFAULT_OUTPUT_ROOT
EXPERIMENT_ROOT.mkdir(exist_ok=True)

# ---- Training regime toggle (pure ratios, NO annealing) ----
# Run the notebook once per regime; checkpoints/outputs are regime-tagged so they
# never collide. This replaces the earlier clean_ratio 0.5->0.1 annealing.
#   "clean"       -> 100% clean examples (clean_ratio 1.0)
#   "adversarial" -> 100% adversarial    (clean_ratio 0.0)
TRAIN_REGIME = "adversarial"   # or "clean"
if TRAIN_REGIME == "clean":
    REGIME_TAG, REGIME_CLEAN_RATIO = "clean", 1.0
elif TRAIN_REGIME == "adversarial":
    REGIME_TAG, REGIME_CLEAN_RATIO = "adv", 0.0
else:
    raise ValueError(f"TRAIN_REGIME must be 'clean' or 'adversarial', got {TRAIN_REGIME!r}")

RUN_TRAINING = True
RUN_ABLATION_TRAINING = False
RUN_ARCHITECTURE_TRAINING = False
RUN_MULTI_SEED = False
RUN_EXTRA_DATASETS = False
PERSIST_OUTPUTS = True
REQUIRE_AUTOATTACK_FOR_FINAL = True
SHOW_NOTEBOOK_PROGRESS = True

RUN_FULL_TEST_THRESHOLD_SWEEP = True
RUN_MULTI_THRESHOLD_ATTACKS = True
RUN_AUTOATTACK_ON_ALL_SELECTED_THRESHOLDS = False
SELECTED_THRESHOLD_COUNT = 3

ATTACK_SUBSET_SAMPLES = 2000
ATTACK_SUBSET_BALANCED = True
ATTACK_SUBSET_SEED = 0

CALIBRATION_BATCH_SIZE = 512
TEST_BATCH_SIZE = 512
ATTACK_BATCH_SIZE = 128
EXACT_VT_BATCH_SIZE = 512

dataset_cfg = DatasetConfig(
    name="cifar10",
    root=DEFAULT_DATA_ROOT,
    batch_size=256,
    num_workers=4,
    seed=0,
    augment=True,
    val_fraction=0.1,
    calibration_fraction=0.1,
    val_subset=None,
    calib_subset=None,
    test_subset=None,
)

model_cfg = ModelConfig(
    architecture="vanilla_resnet18",
    num_classes=10,
    in_channels=3,
    max_time_steps=8,
    input_encoding="direct",
    stem_channels=64,
    stage_channels=(64, 128, 256, 512),
    blocks_per_stage=(2, 2, 2, 2),
    stochastic=False,
    learnable_sigma=False,
    use_dynamic_sigma=False,
    use_shortcut_lif=True,
    v_threshold=0.5,
    lif_tau=2.0,
    detach_reset=True,
    sigma_init_space="raw",
)

training_cfg = TrainingConfig(
    epochs=100,
    base_lr=0.1,
    sigma_lr_scale=0.0,
    momentum=0.9,
    weight_decay=1e-4,
    warmup_epochs=5,
    use_amp=True,
    amp_dtype="float16",
    gradient_clip=1.0,
    clean_ratio_start=REGIME_CLEAN_RATIO,
    clean_ratio_end=REGIME_CLEAN_RATIO,
    entropy_lambda=0.01,
    epsilon=8/255,
    alpha=8/255,
    random_start_scale=0.1,
    use_parseval=False,
    parseval_beta=0.0,
    parseval_interval=5,
    mixed_time_steps=tuple(range(1, model_cfg.max_time_steps + 1)),
    seed=0,
    device="cuda" if torch.cuda.is_available() else "cpu",
    output_dir=str(EXPERIMENT_ROOT / "checkpoints"),
    checkpoint_name=f"vanilla_resnet18_{REGIME_TAG}_full_model.pt",
    eval_interval=5,
    selection_metric="clean",
)

print(f"Training regime: {TRAIN_REGIME} (clean_ratio={REGIME_CLEAN_RATIO}) -> {training_cfg.checkpoint_name}")

eval_cfg = EvaluationConfig(
    vt_thresholds=(0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99),
    fixed_time_steps=tuple(range(1, model_cfg.max_time_steps + 1)),
    compute_metrics=True,
    exact_vt_compute=True,
    match_metric="avg_synops",
    max_batches=None,
    seeds=(0, 1, 2),
)

selection_attack_cfg = AttackConfig(
    epsilon=8/255,
    alpha=2/255,
    steps=4,
    eot_iters=1,
)

adaptive_attack_cfg = AttackConfig(
    epsilon=8/255,
    alpha=2/255,
    steps=20,
    eot_iters=1,
    exit_aware=True,
    force_full_trace=True,
    attack_time_steps=model_cfg.max_time_steps,
)

compute_attack_cfg = replace(
    adaptive_attack_cfg,
    objective="compute_overuse",
    preserve_label_weight=1.0,
)

autoattack_cfg = AttackConfig(
    attack_name="autoattack",
    epsilon=8/255,
    exit_aware=adaptive_attack_cfg.exit_aware,
    force_full_trace=adaptive_attack_cfg.force_full_trace,
    attack_time_steps=model_cfg.max_time_steps,
    eot_iters=1,
    autoattack_version="standard",
)

print(dataset_cfg)
print(model_cfg)
print(training_cfg)
print(f"TF32 matmul: {torch.backends.cuda.matmul.allow_tf32 if torch.cuda.is_available() else 'cpu'}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark if torch.cuda.is_available() else 'cpu'}")
print(f"Main-table VT compute mode: {'sample_exact' if eval_cfg.exact_vt_compute else 'batched_proxy'}")
print(f"Main-table fixed-T match metric: {eval_cfg.match_metric}")
print(f"AutoAttack version policy: {autoattack_cfg.autoattack_version}")
print(f"Selected-threshold panel size: {SELECTED_THRESHOLD_COUNT}")
print(
    f"Eval loader overrides -> calib: {CALIBRATION_BATCH_SIZE or dataset_cfg.batch_size}, "
    f"test: {TEST_BATCH_SIZE or dataset_cfg.batch_size}, "
    f"attack: {ATTACK_BATCH_SIZE or TEST_BATCH_SIZE or dataset_cfg.batch_size}, "
    f"exact_vt: {EXACT_VT_BATCH_SIZE or CALIBRATION_BATCH_SIZE or TEST_BATCH_SIZE or dataset_cfg.batch_size}"
)
print(
    f"Attack subset -> samples: {ATTACK_SUBSET_SAMPLES}, "
    f"balanced: {ATTACK_SUBSET_BALANCED}, seed: {ATTACK_SUBSET_SEED}"
)


In [ ]:
seed_everything(training_cfg.seed)
splits = build_data_splits(dataset_cfg)

def load_or_train_model(current_model_cfg, current_training_cfg, current_splits):
    seed_everything(current_training_cfg.seed)
    model = build_model(current_model_cfg, seed=current_training_cfg.seed).to(current_training_cfg.device)
    checkpoint_path = Path(current_training_cfg.output_dir) / current_training_cfg.checkpoint_name
    history = None
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=current_training_cfg.device)
        model.load_state_dict(checkpoint["model_state_dict"])
        history = checkpoint.get("history")
        print(f"Loaded checkpoint: {checkpoint_path}")
    elif RUN_TRAINING:
        history, checkpoint_path = fit_model(
            model,
            current_splits.train_loader,
            current_splits.val_loader,
            current_training_cfg,
            selection_attack_config=selection_attack_cfg,
        )
        print(f"Trained and saved: {checkpoint_path}")
    else:
        print(f"Checkpoint not found: {checkpoint_path}")
        print("Set RUN_TRAINING = True to train before running the sweeps.")
    return model, history, checkpoint_path

model, history, checkpoint_path = load_or_train_model(model_cfg, training_cfg, splits)


In [ ]:
from importlib.util import find_spec

try:
    from tqdm.auto import tqdm as notebook_tqdm
except ImportError:
    notebook_tqdm = None


def progress_iter(iterable, desc, total=None, leave=False):
    if not SHOW_NOTEBOOK_PROGRESS or notebook_tqdm is None:
        return iterable
    resolved_total = total
    if resolved_total is None:
        try:
            resolved_total = len(iterable)
        except TypeError:
            resolved_total = None
    return notebook_tqdm(iterable, total=resolved_total, desc=desc, leave=leave, dynamic_ncols=True)


def persist_frame(frame, stem):
    if not PERSIST_OUTPUTS or frame is None:
        return None
    path = EXPERIMENT_ROOT / f"{stem}.csv"
    frame.to_csv(path, index=False)
    print(f"Saved {path}")
    return path


def require_autoattack_for_final_runs():
    if not REQUIRE_AUTOATTACK_FOR_FINAL:
        return
    if find_spec("torchattacks") is None and find_spec("autoattack") is None:
        raise RuntimeError(
            "AutoAttack is required for final runs but neither `torchattacks` nor `autoattack` is installed."
        )


def resolve_eval_loader(base_loader, batch_size_override=None):
    if batch_size_override is None or batch_size_override == base_loader.batch_size:
        return base_loader
    return rebuild_loader(base_loader, batch_size=batch_size_override, shuffle=False)


def get_calibration_loaders(current_splits):
    fixed_loader = resolve_eval_loader(current_splits.calib_loader, CALIBRATION_BATCH_SIZE)
    vt_batch_size = EXACT_VT_BATCH_SIZE if eval_cfg.exact_vt_compute else CALIBRATION_BATCH_SIZE
    vt_loader = resolve_eval_loader(current_splits.calib_loader, vt_batch_size)
    return {"fixed": fixed_loader, "vt": vt_loader}


def resolve_attack_loader(base_loader):
    attack_batch_size = ATTACK_BATCH_SIZE if ATTACK_BATCH_SIZE is not None else TEST_BATCH_SIZE
    attack_loader = resolve_eval_loader(base_loader, attack_batch_size)
    if ATTACK_SUBSET_SAMPLES is None:
        return attack_loader
    return build_subset_loader(
        attack_loader,
        ATTACK_SUBSET_SAMPLES,
        ATTACK_SUBSET_SEED,
        balanced=ATTACK_SUBSET_BALANCED,
        batch_size=attack_batch_size,
        shuffle=False,
    )


def get_test_loaders(current_splits):
    fixed_loader = resolve_eval_loader(current_splits.test_loader, TEST_BATCH_SIZE)
    vt_batch_size = EXACT_VT_BATCH_SIZE if eval_cfg.exact_vt_compute else TEST_BATCH_SIZE
    vt_loader = resolve_eval_loader(current_splits.test_loader, vt_batch_size)
    attack_loader = resolve_attack_loader(current_splits.test_loader)
    return {
        "fixed": fixed_loader,
        "vt": vt_loader,
        "attack": attack_loader,
        "diagnostics": attack_loader,
    }


def print_eval_loader_plan(current_splits):
    calibration_loaders = get_calibration_loaders(current_splits)
    test_loaders = get_test_loaders(current_splits)
    print(f"Calibration fixed/batched batch size: {calibration_loaders['fixed'].batch_size}")
    print(f"Calibration VT batch size: {calibration_loaders['vt'].batch_size}")
    print(f"Test fixed/batched batch size: {test_loaders['fixed'].batch_size}")
    print(f"Test VT batch size: {test_loaders['vt'].batch_size}")
    print(f"Attack/diagnostic batch size: {test_loaders['attack'].batch_size}")
    print(f"Attack/diagnostic samples: {len(test_loaders['attack'].dataset)}")
    if ATTACK_SUBSET_SAMPLES is not None:
        subset_mode = "class-balanced" if ATTACK_SUBSET_BALANCED else "random"
        print(f"Attack subset mode: {subset_mode} (seed={ATTACK_SUBSET_SEED})")
    if eval_cfg.exact_vt_compute:
        print(
            "Exact VT compute remains sample-exact inside each batch, so larger VT batch sizes help staging more than GPU saturation."
        )


def calibrate_compute_matched_threshold(current_model, current_splits, current_model_cfg, current_training_cfg, current_stop_rule, prefix, current_attack_config=None):
    calibration_loaders = get_calibration_loaders(current_splits)
    calibration = select_compute_matched_threshold(
        current_model,
        calibration_loaders["vt"],
        thresholds=eval_cfg.vt_thresholds,
        fixed_time_steps=eval_cfg.fixed_time_steps,
        stop_rule=current_stop_rule,
        time_steps=current_model_cfg.max_time_steps,
        attack_config=current_attack_config,
        compute_metrics=eval_cfg.compute_metrics,
        exact_vt_compute=eval_cfg.exact_vt_compute,
        match_metric=eval_cfg.match_metric,
        device=current_training_cfg.device,
        max_batches=eval_cfg.max_batches,
        vt_dataloader=calibration_loaders["vt"],
        fixed_dataloader=calibration_loaders["fixed"],
    )
    persist_frame(calibration["vt_results"], f"{prefix}_vt_threshold_sweep")
    persist_frame(calibration["fixed_results"], f"{prefix}_fixed_t_sweep")
    persist_frame(calibration["matches"], f"{prefix}_vt_fixed_matches")
    return calibration


print_eval_loader_plan(splits)


## 1. Calibration: Fixed-$T$ Controls and VT Thresholds

Calibration happens on the held-out calibration split, not the test split.

In [ ]:
vt_rule = build_stop_rule("max_softmax")
main_calibration = calibrate_compute_matched_threshold(model, splits, model_cfg, training_cfg, vt_rule, "main")
calib_vt = main_calibration["vt_results"]
calib_fixed = main_calibration["fixed_results"]
calib_matches = main_calibration["matches"]

display(calib_fixed[["time_steps", "accuracy", "avg_executed_steps", "avg_synops"]])
display(calib_vt[["threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "compute_mode"]])
display(calib_matches[["threshold", "vt_accuracy", "vt_avg_stopping_depth", "vt_avg_executed_steps", "matched_fixed_t", "fixed_accuracy", "synops_delta", "accuracy_delta"]].sort_values(["accuracy_delta", "synops_delta"], ascending=[False, True]))

selected_threshold = float(main_calibration["selected_threshold"])
selected_synops = float(main_calibration["selected_metric_value"])
selected_threshold_panel = select_threshold_neighbors(calib_vt, selected_threshold, count=SELECTED_THRESHOLD_COUNT)
selected_threshold_values = selected_threshold_panel["threshold"].tolist()
persist_frame(selected_threshold_panel, "main_selected_thresholds")

print(f"Selected threshold on calibration split: {selected_threshold:.2f}")
print(f"Matched fixed-T baseline from calibration: T={int(main_calibration['selected_match']['matched_fixed_t'])}")
print(f"Target SynOps for matched stopping-rule calibration: {selected_synops:.2f}")
display(selected_threshold_panel)


## 2. Fixed-$T$ Baseline then VT Test Evaluation


In [ ]:
test_loaders = get_test_loaders(splits)
selected_fixed_t = int(main_calibration["selected_match"]["matched_fixed_t"])

print("Running fixed-T baseline suite on test first...")
test_fixed_sweep = fixed_time_sweep(
    model,
    test_loaders["fixed"],
    time_step_values=eval_cfg.fixed_time_steps,
    compute_metrics=True,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
persist_frame(test_fixed_sweep, "test_fixed_t_sweep")

test_fixed_full = test_fixed_sweep.loc[test_fixed_sweep["time_steps"] == model_cfg.max_time_steps].iloc[0].to_dict()
test_fixed_full["report_mode"] = "fixed_full_horizon"

test_fixed_matched = test_fixed_sweep.loc[test_fixed_sweep["time_steps"] == selected_fixed_t].iloc[0].to_dict()
test_fixed_matched["report_mode"] = "fixed_compute_matched"

fixed_baseline_df = pd.DataFrame([test_fixed_full, test_fixed_matched])
persist_frame(fixed_baseline_df, "test_fixed_baseline_results")
display(fixed_baseline_df[["report_mode", "mode", "time_steps", "accuracy", "avg_executed_steps", "avg_synops", "avg_spike_count", "compute_mode"]])

print("Running VT suite on test for comparison...")
if RUN_FULL_TEST_THRESHOLD_SWEEP:
    test_vt_sweep = threshold_sweep(
        model,
        test_loaders["vt"],
        thresholds=eval_cfg.vt_thresholds,
        stop_rule=vt_rule,
        time_steps=model_cfg.max_time_steps,
        compute_metrics=True,
        exact_vt_compute=eval_cfg.exact_vt_compute,
        device=training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )
    test_vt_matches = match_fixed_time_controls(
        test_vt_sweep,
        test_fixed_sweep,
        match_metric=eval_cfg.match_metric,
    )

    persist_frame(test_vt_sweep, "test_vt_threshold_sweep")
    persist_frame(test_vt_matches, "test_vt_fixed_matches")

    selected_threshold_test_panel = selected_threshold_panel.merge(
        test_vt_sweep,
        on="threshold",
        how="left",
    )
else:
    selected_threshold_rows = []
    for _, threshold_row in selected_threshold_panel.iterrows():
        result = evaluate_policy(
            model,
            test_loaders["vt"],
            mode="vt",
            time_steps=model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=float(threshold_row["threshold"]),
            compute_metrics=True,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=training_cfg.device,
            max_batches=eval_cfg.max_batches,
        )
        result["threshold_role"] = threshold_row["threshold_role"]
        selected_threshold_rows.append(result)
    selected_threshold_test_panel = pd.DataFrame(selected_threshold_rows)

persist_frame(selected_threshold_test_panel, "test_selected_threshold_results")
test_vt = selected_threshold_test_panel.loc[selected_threshold_test_panel["threshold_role"] == "selected"].iloc[0].to_dict()
test_vt["report_mode"] = "vt_selected_threshold"

main_test_df = pd.DataFrame([test_fixed_full, test_fixed_matched, test_vt])
persist_frame(main_test_df, "main_test_results")
display(main_test_df[["report_mode", "mode", "time_steps", "threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "avg_spike_count", "compute_mode"]])
display(selected_threshold_test_panel[["threshold_role", "threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "compute_mode"]])

if RUN_FULL_TEST_THRESHOLD_SWEEP:
    display(test_vt_sweep[["threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "compute_mode"]])
    display(
        test_vt_matches[
            [
                "threshold",
                "vt_accuracy",
                "vt_avg_stopping_depth",
                "vt_avg_executed_steps",
                "matched_fixed_t",
                "fixed_accuracy",
                "synops_delta",
                "accuracy_delta",
            ]
        ].sort_values(["accuracy_delta", "synops_delta"], ascending=[False, True])
    )


## 3. Stopping-Rule Comparison Against the Fixed-$T$ Baseline


In [ ]:
calibration_loaders = get_calibration_loaders(splits)
test_loaders = get_test_loaders(splits)

rule_calibration = compare_stopping_rules(
    model,
    calibration_loaders["vt"],
    rule_names=eval_cfg.stopping_rules,
    candidate_thresholds=eval_cfg.vt_thresholds,
    time_steps=model_cfg.max_time_steps,
    target_value=selected_synops,
    compute_metrics=True,
    exact_vt_compute=eval_cfg.exact_vt_compute,
    target_metric=eval_cfg.match_metric,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
persist_frame(rule_calibration, "stopping_rule_calibration")
display(rule_calibration[["stop_rule", "threshold", "accuracy", "avg_synops", "metric_gap"]])

fixed_rule_anchor = test_fixed_full.copy()
fixed_rule_anchor["stop_rule"] = "fixed_T8_baseline"
fixed_rule_anchor["threshold"] = None

rule_test_rows = [fixed_rule_anchor]
for _, row in rule_calibration.iterrows():
    result = evaluate_policy(
        model,
        test_loaders["vt"],
        mode="vt",
        time_steps=model_cfg.max_time_steps,
        stop_rule=build_stop_rule(row["stop_rule"]),
        threshold=float(row["threshold"]),
        compute_metrics=True,
        exact_vt_compute=eval_cfg.exact_vt_compute,
        device=training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )
    result["stop_rule"] = row["stop_rule"]
    rule_test_rows.append(result)

rule_test_df = pd.DataFrame(rule_test_rows)
persist_frame(rule_test_df, "stopping_rule_test_results")
display(rule_test_df[["stop_rule", "threshold", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops"]].sort_values(["stop_rule", "accuracy"], ascending=[True, False]))


## 4. Fixed-$T$ Baseline and VT Attacks


In [ ]:
attack_rows = []
require_autoattack_for_final_runs()
test_loaders = get_test_loaders(splits)
attack_threshold_panel = selected_threshold_panel if RUN_MULTI_THRESHOLD_ATTACKS else selected_threshold_panel[selected_threshold_panel["threshold_role"] == "selected"]
attack_objectives = ["untargeted", "wrong_early_exit", "compute_overuse"]

print("Running fixed-T baseline attacks first...")
fixed_attack = evaluate_policy(
    model,
    test_loaders["attack"],
    mode="fixed",
    time_steps=model_cfg.max_time_steps,
    attack_config=replace(selection_attack_cfg, steps=20, eot_iters=8),
    compute_metrics=True,
    exact_vt_compute=False,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
fixed_attack["threshold_role"] = "fixed_T8_baseline"
attack_rows.append(fixed_attack)

print("Running VT attacks against the selected threshold panel...")
for objective in progress_iter(attack_objectives, desc="Attack objectives", total=len(attack_objectives), leave=False):
    current_attack = replace(adaptive_attack_cfg if objective != "compute_overuse" else compute_attack_cfg, objective=objective)
    for _, threshold_row in attack_threshold_panel.iterrows():
        result = evaluate_policy(
            model,
            test_loaders["attack"],
            mode="vt",
            time_steps=model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=float(threshold_row["threshold"]),
            attack_config=current_attack,
            compute_metrics=True,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=training_cfg.device,
            max_batches=eval_cfg.max_batches,
        )
        result["threshold_role"] = threshold_row["threshold_role"]
        attack_rows.append(result)

try:
    autoattack_fixed = evaluate_policy(
        model,
        test_loaders["attack"],
        mode="fixed",
        time_steps=model_cfg.max_time_steps,
        attack_config=autoattack_cfg,
        compute_metrics=False,
        exact_vt_compute=False,
        device=training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )
    autoattack_fixed["threshold_role"] = "fixed_T8_baseline"
    attack_rows.append(autoattack_fixed)

    vt_autoattack_panel = attack_threshold_panel if RUN_AUTOATTACK_ON_ALL_SELECTED_THRESHOLDS else attack_threshold_panel[attack_threshold_panel["threshold_role"] == "selected"]
    for _, threshold_row in vt_autoattack_panel.iterrows():
        autoattack_vt = evaluate_policy(
            model,
            test_loaders["attack"],
            mode="vt",
            time_steps=model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=float(threshold_row["threshold"]),
            attack_config=autoattack_cfg,
            compute_metrics=False,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=training_cfg.device,
            max_batches=eval_cfg.max_batches,
        )
        autoattack_vt["threshold_role"] = threshold_row["threshold_role"]
        attack_rows.append(autoattack_vt)
except ImportError as exc:
    print(f"AutoAttack skipped: {exc}")

attack_df = pd.DataFrame(attack_rows)
persist_frame(attack_df, "attack_results")
display(attack_df[["mode", "threshold_role", "threshold", "attack_name", "autoattack_version", "attack_objective", "accuracy", "avg_stopping_depth", "avg_executed_steps", "avg_synops", "attack_eot"]])

fixed_clean_diag = collect_temporal_diagnostics(
    model,
    test_loaders["diagnostics"],
    time_steps=model_cfg.max_time_steps,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
vt_clean_diag = collect_temporal_diagnostics(
    model,
    test_loaders["diagnostics"],
    time_steps=model_cfg.max_time_steps,
    stop_rule=vt_rule,
    threshold=selected_threshold,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)
vt_attack_diag = collect_temporal_diagnostics(
    model,
    test_loaders["diagnostics"],
    time_steps=model_cfg.max_time_steps,
    stop_rule=vt_rule,
    threshold=selected_threshold,
    attack_config=compute_attack_cfg,
    device=training_cfg.device,
    max_batches=eval_cfg.max_batches,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
steps = range(1, len(vt_clean_diag["avg_confidence_by_step"]) + 1)
axes[0].plot(steps, fixed_clean_diag["avg_confidence_by_step"], label="Fixed T=8")
axes[0].plot(steps, vt_clean_diag["avg_confidence_by_step"], label="VT Clean")
axes[0].plot(steps, vt_attack_diag["avg_confidence_by_step"], label="VT Compute Attack")
axes[0].set_title("Confidence Over Time")
axes[0].set_xlabel("Step")
axes[0].legend()
axes[1].plot(steps, fixed_clean_diag["avg_accuracy_by_step"], label="Fixed T=8")
axes[1].plot(steps, vt_clean_diag["avg_accuracy_by_step"], label="VT Clean")
axes[1].plot(steps, vt_attack_diag["avg_accuracy_by_step"], label="VT Compute Attack")
axes[1].set_title("Accuracy Over Time")
axes[1].set_xlabel("Step")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Fixed-T clean exit histogram:", fixed_clean_diag["step_histogram"])
print("VT clean exit histogram:", vt_clean_diag["step_histogram"])
print("VT compute-attack exit histogram:", vt_attack_diag["step_histogram"])


## 5. Minimal Ablation Matrix

Each variant is selected on validation, optionally calibrated on the calibration split, and only then reported on test.

In [ ]:
ablation_rows = []
ablation_dir = EXPERIMENT_ROOT / "ablations"
ablation_dir.mkdir(exist_ok=True)
ablation_specs = default_ablation_specs(model_cfg.max_time_steps)
test_loaders = get_test_loaders(splits)

for spec in progress_iter(ablation_specs, desc="Ablations", total=len(ablation_specs), leave=False):
    variant_model_cfg, variant_training_cfg = apply_ablation(model_cfg, training_cfg, spec)
    variant_training_cfg = replace(variant_training_cfg, output_dir=str(ablation_dir), checkpoint_name=f"{spec.name}.pt")
    seed_everything(variant_training_cfg.seed)
    variant_model = build_model(variant_model_cfg, seed=variant_training_cfg.seed).to(variant_training_cfg.device)
    variant_checkpoint = Path(variant_training_cfg.output_dir) / variant_training_cfg.checkpoint_name
    if variant_checkpoint.exists():
        checkpoint = torch.load(variant_checkpoint, map_location=variant_training_cfg.device)
        variant_model.load_state_dict(checkpoint["model_state_dict"])
    elif RUN_ABLATION_TRAINING:
        fit_model(variant_model, splits.train_loader, splits.val_loader, variant_training_cfg, selection_attack_config=selection_attack_cfg)
    else:
        print(f"Skipping {spec.name}; checkpoint not found.")
        continue

    fixed_eval = evaluate_policy(
        variant_model,
        test_loaders["fixed"],
        mode="fixed",
        time_steps=variant_model_cfg.max_time_steps,
        compute_metrics=True,
        exact_vt_compute=False,
        device=variant_training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )

    vt_eval = None
    chosen_threshold = None
    if spec.vt_enabled:
        ablation_calibration = calibrate_compute_matched_threshold(
            variant_model,
            splits,
            variant_model_cfg,
            variant_training_cfg,
            vt_rule,
            f"ablation_{spec.name}",
        )
        chosen_threshold = float(ablation_calibration["selected_threshold"])
        vt_eval = evaluate_policy(
            variant_model,
            test_loaders["vt"],
            mode="vt",
            time_steps=variant_model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=chosen_threshold,
            compute_metrics=True,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=variant_training_cfg.device,
            max_batches=eval_cfg.max_batches,
        )

    ablation_rows.append({
        "variant": spec.name,
        "selected_threshold": chosen_threshold,
        "fixed_accuracy": fixed_eval["accuracy"],
        "fixed_synops": fixed_eval["avg_synops"],
        "vt_accuracy": None if vt_eval is None else vt_eval["accuracy"],
        "vt_avg_executed_steps": None if vt_eval is None else vt_eval["avg_executed_steps"],
        "vt_synops": None if vt_eval is None else vt_eval["avg_synops"],
    })

ablation_df = pd.DataFrame(ablation_rows)
persist_frame(ablation_df, "ablation_results")
display(ablation_df)


## 6. Cross-Architecture Sweep

This uses the same train / val / calib / test pipeline for ResNet, VGG, and WRN.

In [ ]:
architecture_presets = {
    "resnet": replace(model_cfg, architecture="resnet", stem_channels=128, stage_channels=(128, 256, 512), blocks_per_stage=(3, 3, 2)),
    "vgg": replace(model_cfg, architecture="vgg", stem_channels=64, stage_channels=(64, 128, 256, 512, 512), blocks_per_stage=(1, 1, 2, 2, 2), dropout_rate=0.5),
    "wrn": replace(model_cfg, architecture="wrn", stem_channels=16, widen_factor=10, wrn_depth=28, stage_channels=(160, 320, 640), blocks_per_stage=(4, 4, 4)),
}

test_loaders = get_test_loaders(splits)
architecture_rows = []
architecture_items = list(architecture_presets.items())
for name, arch_model_cfg in progress_iter(architecture_items, desc="Architectures", total=len(architecture_items), leave=False):
    arch_training_cfg = replace(training_cfg, checkpoint_name=f"{name}_full_model.pt")
    seed_everything(arch_training_cfg.seed)
    arch_model = build_model(arch_model_cfg, seed=arch_training_cfg.seed).to(arch_training_cfg.device)
    arch_checkpoint = Path(arch_training_cfg.output_dir) / arch_training_cfg.checkpoint_name

    if arch_checkpoint.exists():
        checkpoint = torch.load(arch_checkpoint, map_location=arch_training_cfg.device)
        arch_model.load_state_dict(checkpoint["model_state_dict"])
    elif RUN_ARCHITECTURE_TRAINING:
        fit_model(arch_model, splits.train_loader, splits.val_loader, arch_training_cfg, selection_attack_config=selection_attack_cfg)
    else:
        print(f"Skipping {name}; checkpoint not found.")
        continue

    arch_calibration = calibrate_compute_matched_threshold(
        arch_model,
        splits,
        arch_model_cfg,
        arch_training_cfg,
        vt_rule,
        f"architecture_{name}",
    )
    arch_threshold = float(arch_calibration["selected_threshold"])

    arch_fixed = evaluate_policy(
        arch_model,
        test_loaders["fixed"],
        mode="fixed",
        time_steps=arch_model_cfg.max_time_steps,
        compute_metrics=True,
        exact_vt_compute=False,
        device=arch_training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )
    arch_vt = evaluate_policy(
        arch_model,
        test_loaders["vt"],
        mode="vt",
        time_steps=arch_model_cfg.max_time_steps,
        stop_rule=vt_rule,
        threshold=arch_threshold,
        compute_metrics=True,
        exact_vt_compute=eval_cfg.exact_vt_compute,
        device=arch_training_cfg.device,
        max_batches=eval_cfg.max_batches,
    )

    architecture_rows.append({
        "architecture": name,
        "selected_threshold": arch_threshold,
        "fixed_accuracy": arch_fixed["accuracy"],
        "fixed_synops": arch_fixed["avg_synops"],
        "vt_accuracy": arch_vt["accuracy"],
        "vt_avg_executed_steps": arch_vt["avg_executed_steps"],
        "vt_synops": arch_vt["avg_synops"],
    })

architecture_df = pd.DataFrame(architecture_rows)
persist_frame(architecture_df, "architecture_results")
display(architecture_df)


## 7. Optional Multi-Seed Summary

This is intentionally left opt-in because the full paper run is expensive.

In [ ]:
if RUN_MULTI_SEED:
    multiseed_rows = []
    seed_values = list(eval_cfg.seeds)
    for seed in progress_iter(seed_values, desc="Seeds", total=len(seed_values), leave=False):
        seed_dataset_cfg = replace(dataset_cfg, seed=seed)
        seed_splits = build_data_splits(seed_dataset_cfg)
        seed_train_cfg = replace(training_cfg, seed=seed, checkpoint_name=f"{model_cfg.architecture}_seed{seed}.pt")
        seed_everything(seed)
        seed_model = build_model(model_cfg, seed=seed).to(seed_train_cfg.device)
        seed_checkpoint = Path(seed_train_cfg.output_dir) / seed_train_cfg.checkpoint_name
        if seed_checkpoint.exists():
            checkpoint = torch.load(seed_checkpoint, map_location=seed_train_cfg.device)
            seed_model.load_state_dict(checkpoint["model_state_dict"])
        else:
            fit_model(seed_model, seed_splits.train_loader, seed_splits.val_loader, seed_train_cfg, selection_attack_config=selection_attack_cfg)

        seed_calibration = calibrate_compute_matched_threshold(
            seed_model,
            seed_splits,
            model_cfg,
            seed_train_cfg,
            vt_rule,
            f"multiseed_seed{seed}",
        )
        seed_threshold = float(seed_calibration["selected_threshold"])
        seed_test_loaders = get_test_loaders(seed_splits)
        seed_eval = evaluate_policy(
            seed_model,
            seed_test_loaders["vt"],
            mode="vt",
            time_steps=model_cfg.max_time_steps,
            stop_rule=vt_rule,
            threshold=seed_threshold,
            compute_metrics=True,
            exact_vt_compute=eval_cfg.exact_vt_compute,
            device=seed_train_cfg.device,
            max_batches=eval_cfg.max_batches,
        )
        seed_eval["selected_threshold"] = seed_threshold
        seed_eval["target_synops"] = float(seed_calibration["selected_metric_value"])
        seed_eval["seed"] = seed
        seed_eval["architecture"] = model_cfg.architecture
        multiseed_rows.append(seed_eval)

    multiseed_df = pd.DataFrame(multiseed_rows)
    multiseed_summary = summarize_multiseed(multiseed_rows, group_cols=["architecture"])
    persist_frame(multiseed_df, "multiseed_results")
    persist_frame(multiseed_summary, "multiseed_summary")
    display(multiseed_df)
    display(multiseed_summary)
else:
    print("Set RUN_MULTI_SEED = True to run the multi-seed summary.")
